# 🌍🌙 Agent IA « Lumières nocturnes » — analyser les NTL d'un pays avec Groq, Gemini ou Ollama

**Atelier STG17** · *Emerging Issues, Emerging Practice* — Jour 1 (*From RAG to Agent*) × Jour 4 (*Africa after dark*)

Ce notebook construit, étape par étape, un **agent IA** capable de répondre en langage naturel à des questions sur les **lumières nocturnes (Night-Time Lights, NTL)** d'un pays identifié par son **code ISO3**. L'agent s'appuie sur une **boîte à outils de 21 fonctions** (séries temporelles, statistiques infranationales, détection d'anomalies, population non éclairée, validation avec la Banque mondiale, impact d'événements, contrôle qualité, cartes et graphiques) alimentée par **Google Earth Engine**.

| Fonctionne sur | Cerveau (LLM) | Données |
|---|---|---|
| ✅ Google Colab · ✅ Kaggle · ✅ Jupyter local / VS Code | Groq · Gemini · Ollama (via **une seule** interface compatible OpenAI) | VIIRS (NOAA/EOG) · Black Marble (NASA) · WorldPop · geoBoundaries · API Banque mondiale |

**Plan du notebook**

| # | Section | Ce que vous apprenez |
|---|---|---|
| 00 | Mode d'emploi | Clés, comptes et prérequis selon l'environnement |
| 01 | Paramètres | Choisir le pays, le fournisseur LLM et la source de données |
| 02–03 | Installation · identité visuelle | Un notebook portable et une charte graphique cohérente |
| 04 | Le cerveau : LLM | Une interface unique pour trois fournisseurs |
| 05 | Les données NTL | Produits VIIRS, artefacts, Earth Engine ou mode démo |
| 06 | La boîte à outils | Concevoir des outils qu'un LLM peut utiliser sans se tromper |
| 07 | Tester les outils à la main | Comprendre ce que « voit » l'agent |
| 08 | L'agent | La boucle *raisonner → agir → observer* et la mise en forme des réponses |
| 09–10 | Démonstrations · Chat | Questions guidées puis conversation libre |
| 11–12 | Benchmark · Évaluation | Comparer les fournisseurs, mesurer la fiabilité |
| 13 | Publication | Rapport HTML prêt pour GitHub Pages |
| 14–15 | Extension · Limites | Ajouter vos outils, rester honnête sur ce qu'un proxy ne permet pas |

> 💡 **Exécutez les cellules dans l'ordre** (`Exécution › Tout exécuter` fonctionne aussi). Seules les cellules 01 (paramètres) et 09 (questions) sont à modifier.

In [ ]:
#@title 🌙 Couverture
from IPython.display import HTML, display
display(HTML("""
<div style="position:relative;overflow:hidden;border-radius:16px;padding:40px 44px 52px;
            background:linear-gradient(135deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;color:#fff;">
  <svg style="position:absolute;right:-70px;top:-80px;opacity:.16" width="460" height="460" viewBox="0 0 460 460">
    <g fill="none" stroke="#fff" stroke-width="2">
      <circle cx="230" cy="230" r="220"/><circle cx="230" cy="230" r="165"/>
      <circle cx="230" cy="230" r="110"/><circle cx="230" cy="230" r="55"/></g></svg>
  <svg style="position:absolute;right:40px;bottom:30px" width="330" height="150" viewBox="0 0 330 150">
    <g fill="#F5C242">
      <circle cx="40" cy="110" r="3" opacity=".9"/><circle cx="70" cy="95" r="2" opacity=".6"/>
      <circle cx="95" cy="120" r="5" opacity=".95"/><circle cx="130" cy="70" r="2.5" opacity=".7"/>
      <circle cx="160" cy="100" r="7" opacity="1"/><circle cx="170" cy="92" r="3" opacity=".8"/>
      <circle cx="205" cy="60" r="2" opacity=".5"/><circle cx="235" cy="115" r="4" opacity=".9"/>
      <circle cx="260" cy="80" r="2" opacity=".6"/><circle cx="290" cy="105" r="3" opacity=".8"/>
      <circle cx="120" cy="40" r="1.5" opacity=".5"/><circle cx="310" cy="50" r="1.5" opacity=".5"/></g></svg>
  <div style="letter-spacing:3px;font-size:12px;font-weight:700;color:#F5C242;text-transform:uppercase">
    STG17 · Technical Workshop · Africa after dark</div>
  <div style="font-size:42px;font-weight:700;margin:12px 0 6px;line-height:1.08">Agent IA « Lumières nocturnes »</div>
  <div style="font-size:18px;font-style:italic;color:#E6F6EE;max-width:720px">
    Un analyste géospatial augmenté : du code ISO d'un pays à des indicateurs infranationaux,
    des cartes et une note méthodologique — en langage naturel.</div>
  <div style="margin-top:22px;display:flex;flex-wrap:wrap;gap:8px">
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">🧠 Groq · Gemini · Ollama</span>
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">🛰️ Google Earth Engine · VIIRS · Black Marble</span>
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">🧰 21 outils</span>
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">💻 Colab · Kaggle · Local</span>
  </div>
  <div style="position:absolute;left:0;right:0;bottom:0;height:8px;background:#F5C242"></div>
</div>
<div style="font-family:Calibri,Carlito,Arial,sans-serif;font-size:11px;color:#8A9590;margin-top:4px">
  Charte graphique inspirée de la BAD (non officielle) · Notebook pédagogique — atelier STG17</div>
"""))

## 00 · Mode d'emploi

### Ce qu'il vous faut

| Élément | Obligatoire ? | Où l'obtenir |
|---|---|---|
| **Une clé LLM** — `GROQ_API_KEY` *ou* `GEMINI_API_KEY` | Oui, sauf Ollama | Groq : <https://console.groq.com/keys> · Gemini : <https://aistudio.google.com/apikey> (niveaux gratuits disponibles) |
| **Un projet Google Cloud enregistré pour Earth Engine** | Recommandé | <https://code.earthengine.google.com/register> (usage non commercial gratuit), puis indiquez l'identifiant du projet dans `GEE_PROJECT` |
| Ollama | Seulement si `LLM_PROVIDER="ollama"` | Local : <https://ollama.com/download> · Colab/Kaggle : installé automatiquement par la section 04 (GPU conseillé) |

> 🧪 **Pas encore de compte Earth Engine ?** Laissez `DATA_BACKEND="auto"` : sans projet GEE, le notebook bascule en **mode démo** avec des données **synthétiques** (clairement signalées partout) pour que vous puissiez apprendre la mécanique de l'agent. Les chiffres du mode démo ne doivent **jamais** être cités.

### Où ranger les clés (jamais en clair dans le notebook)

| Environnement | Méthode |
|---|---|
| **Colab** | Icône 🔑 *Secrets* (barre de gauche) → ajoutez `GROQ_API_KEY` / `GEMINI_API_KEY` / `GEE_PROJECT` et activez « Accès au notebook ». Earth Engine : une fenêtre d'autorisation Google s'ouvre au premier lancement. |
| **Kaggle** | *Add-ons › Secrets* → mêmes noms. Activez **Internet** dans le panneau *Settings*. Pour Earth Engine, ajoutez de préférence un secret `GEE_SERVICE_ACCOUNT_JSON` (clé JSON d'un compte de service enregistré pour Earth Engine) ; sinon une authentification par lien/code vous sera proposée. |
| **Local** | Variables d'environnement ou fichier `.env` à côté du notebook (`GROQ_API_KEY=...`). Earth Engine : `earthengine authenticate` une fois dans un terminal. |

Si une clé est introuvable, le notebook vous la demande dans un champ masqué (`getpass`).

## 01 · Paramètres de la session

Choisissez **le pays** (code ISO 3166-1 alpha-3 : `RWA`, `SEN`, `CIV`, `TUN`, `KEN`, `NGA`, `MOZ`, `CMR`, `SOM`…), **le fournisseur** et **la source de données**.
Sur Colab, ces champs apparaissent comme un formulaire.

| Paramètre | Rôle |
|---|---|
| `LLM_MODEL` | Vide = modèle par défaut du fournisseur (voir section 04). Le modèle doit savoir **appeler des outils** (*function calling*). |
| `TOOLSET` | `full` = 21 outils ; `core` = 9 outils essentiels, recommandé pour les **petits modèles locaux** (Ollama 7–8B), qui se perdent quand le catalogue est trop long. |
| `LIT_THRESHOLD` | Seuil (nW/cm²/sr) au-delà duquel un pixel est considéré « éclairé ». 0,5 est une valeur courante ; testez-en l'effet. |

In [ ]:
#@title ⚙️ Paramètres de la session
COUNTRY_ISO3    = "RWA"        #@param {type:"string"}
LLM_PROVIDER    = "groq"       #@param ["groq", "gemini", "ollama"]
LLM_MODEL       = ""           #@param {type:"string"}
DATA_BACKEND    = "auto"       #@param ["auto", "gee", "demo"]
GEE_PROJECT     = ""           #@param {type:"string"}
ANSWER_LANGUAGE = "français"   #@param ["français", "English", "português"]
TOOLSET         = "full"       #@param ["full", "core"]
LIT_THRESHOLD   = 0.5          #@param {type:"number"}
MAX_AGENT_STEPS = 12           #@param {type:"integer"}

TEMPERATURE = 0.1                     # faible = réponses plus stables et reproductibles
OUTPUT_DIR  = "ntl_agent_outputs"     # figures, données, cache et rapport
COUNTRY_ISO3 = COUNTRY_ISO3.strip().upper()
print(f"Pays : {COUNTRY_ISO3} · LLM : {LLM_PROVIDER} {LLM_MODEL or '(modèle par défaut)'} · données : {DATA_BACKEND}")

## 02 · Installation et détection de l'environnement

La cellule suivante détecte automatiquement **Colab**, **Kaggle** ou **local**, n'installe que les paquets manquants, puis prépare les dossiers de sortie. Elle est idempotente : vous pouvez la relancer sans risque.

In [ ]:
import os, sys, subprocess, importlib, importlib.util, platform, time, json, re, math, inspect
import hashlib, zlib, pickle, io, base64, difflib, textwrap, warnings, datetime as dt
from pathlib import Path
from typing import Optional, Literal, get_origin, get_args, Union

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
ENV_NAME  = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local")

PACKAGES = {  # module importé -> spécification pip
    "openai": "openai>=1.40", "plotly": "plotly>=5.20", "pycountry": "pycountry",
    "markdown": "markdown", "ee": "earthengine-api", "shapely": "shapely>=2.0",
    "scipy": "scipy", "ipywidgets": "ipywidgets", "requests": "requests",
    "pandas": "pandas", "numpy": "numpy", "openpyxl": "openpyxl", "matplotlib": "matplotlib",
}
missing = [spec for mod, spec in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("📦 Installation :", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import markdown as md_lib
from IPython.display import HTML, Markdown, Image as IPImage, display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_columns", 30)

if IN_COLAB:
    pio.renderers.default = "colab"
elif IN_KAGGLE:
    pio.renderers.default = "notebook"

OUT = Path(OUTPUT_DIR)
for sub in ("figures", "data", "cache"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print(f"✅ Environnement : {ENV_NAME} · Python {platform.python_version()} · sorties dans ./{OUT}")

## 03 · Identité visuelle et composants d'affichage

Un agent utile ne se contente pas de calculer juste : il **présente** ses résultats de façon lisible. On définit ici une fois pour toutes :

* la **palette** (vert institutionnel, or en accent, rouge brique réservé aux baisses et aux risques) et un **thème Plotly** assorti ;
* de petits **composants HTML** (cartes, indicateurs clés, alertes) réutilisés par l'agent.

> Chaque sortie HTML embarque son propre style : dans Colab, chaque cellule s'affiche dans un cadre isolé, une feuille de style globale ne suffirait pas.

In [ ]:
C = dict(green="#00A86A", deep="#00704A", forest="#00553A", gold="#F5C242", ochre="#D49A00",
         teal="#0E7C86", terra="#C4621D", brick="#B83B2E", ink="#231F20", slate="#5E6964",
         mist="#F4F7F5", mint="#E8F5EF", sage="#D5DED9", grey="#A9B5B0")
RAMP   = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
SERIES = [C["green"], C["deep"], C["teal"], C["ochre"], C["terra"], C["brick"], "#7BCBA9", C["grey"]]
NIGHT  = ["#000814", "#061a33", "#0b2e4f", "#1d4e6b", "#b88a1c", "#F5C242", "#FFF3B0", "#FFFFFF"]
FONT   = "Calibri, Carlito, 'Segoe UI', Arial, sans-serif"

pio.templates["stg17"] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=13, color=C["ink"]),
    colorway=SERIES, paper_bgcolor="white", plot_bgcolor="white",
    title=dict(font=dict(size=18, color=C["ink"]), x=0.01, xanchor="left"),
    xaxis=dict(showgrid=False, linecolor=C["sage"], ticks="outside", tickcolor=C["sage"],
               tickfont=dict(color=C["slate"]), title=dict(font=dict(color=C["slate"]))),
    yaxis=dict(gridcolor="#E1E7E4", zeroline=False, linecolor=C["sage"],
               tickfont=dict(color=C["slate"]), title=dict(font=dict(color=C["slate"]))),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=C["slate"])),
    hoverlabel=dict(font=dict(family=FONT)),
    margin=dict(l=60, r=30, t=70, b=80),
))
pio.templates.default = "plotly_white+stg17"

BASE_CSS = """<style>
.ntl{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;color:#231F20;line-height:1.5;font-size:14.5px}
.ntl-card{background:#fff;border:1px solid #D5DED9;border-radius:12px;padding:18px 22px 16px;margin:10px 0;
  position:relative;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,.05)}
.ntl-card:before{content:"";position:absolute;left:0;top:0;right:0;height:4px;
  background:linear-gradient(90deg,#F5C242 0 96px,#00A86A 96px)}
.ntl-card.hl{background:#E8F5EF;border:1.5px solid #00A86A}
.ntl-kicker{font-size:11px;font-weight:700;letter-spacing:3px;text-transform:uppercase;color:#00704A;margin-bottom:4px}
.ntl-title{font-size:20px;font-weight:700;color:#231F20;margin:0 0 6px}
.ntl-muted{color:#5E6964;font-size:12.5px}
.ntl h1,.ntl h2,.ntl h3,.ntl h4{color:#231F20;margin:14px 0 6px;line-height:1.25}
.ntl h3{font-size:17px;border-left:4px solid #00A86A;padding-left:9px}
.ntl table{border-collapse:collapse;margin:8px 0 10px;font-size:13.5px}
.ntl th{background:#E8F5EF;color:#00553A;text-align:left;padding:6px 10px;border-bottom:2px solid #00A86A}
.ntl td{padding:5px 10px;border-bottom:1px solid #D5DED9;vertical-align:top;text-align:left}
.ntl tr:nth-child(even) td{background:#FAFCFB}
.ntl code{background:#F4F7F5;padding:1px 5px;border-radius:4px;font-family:'Courier New',monospace;font-size:12.5px}
.ntl pre{background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:10px;overflow:auto;font-size:12px}
.ntl-kpis{display:flex;flex-wrap:wrap;gap:10px;margin:10px 0 6px}
.ntl-kpi{flex:1 1 160px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:10px 14px}
.ntl-kpi .v{font-size:26px;font-weight:700;color:#00704A;line-height:1.1}
.ntl-kpi .u{font-size:13px;color:#5E6964;font-weight:400}
.ntl-kpi .l{font-size:12px;color:#5E6964;margin-top:2px}
.ntl-kpi .d{font-size:12px;font-weight:700}
.ntl-up{color:#00A86A}.ntl-down{color:#B83B2E}
.ntl-pill{display:inline-block;padding:2px 10px;border-radius:999px;font-size:12px;border:1px solid #00A86A;
  background:#E8F5EF;color:#00553A;margin:2px 4px 2px 0}
.ntl-pill.gold{border-color:#D49A00;background:rgba(245,194,66,.15);color:#6b4d00}
.ntl-pill.red{border-color:#B83B2E;background:rgba(184,59,46,.08);color:#B83B2E}
.ntl-warn{background:rgba(212,154,0,.12);border:1px solid #D49A00;border-radius:10px;padding:10px 14px;margin:8px 0;color:#4d3800}
.ntl-alert{background:rgba(184,59,46,.08);border:1px solid #B83B2E;border-radius:10px;padding:10px 14px;margin:8px 0;color:#7a2219}
.ntl-ok{background:#E8F5EF;border:1px solid #00A86A;border-radius:10px;padding:10px 14px;margin:8px 0;color:#00553A}
.ntl details{margin-top:8px}.ntl summary{cursor:pointer;color:#00704A;font-weight:700}
.ntl-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(250px,1fr));gap:10px}
.ntl-tool{background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:10px 12px}
.ntl-tool b{font-family:'Courier New',monospace;font-size:13px;color:#00553A}
</style>"""


def fmt(x, digits=2):
    """Format lisible : 1 234 567 → '1,23 M' ; 0.1234 → '0.12'."""
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "—"
    if isinstance(x, (int, np.integer)) and abs(x) < 1e6:
        return f"{x:,}".replace(",", " ")
    x = float(x)
    if abs(x) >= 1e9: return f"{x/1e9:.2f} Md"
    if abs(x) >= 1e6: return f"{x/1e6:.2f} M"
    if abs(x) >= 1e3: return f"{x:,.0f}".replace(",", " ")
    return f"{x:.{digits}f}"


class UI:
    """Petits composants HTML réutilisables (tous auto-stylés)."""
    @staticmethod
    def show(body):
        display(HTML(BASE_CSS + f'<div class="ntl">{body}</div>'))

    @staticmethod
    def md(text):
        """Markdown → HTML, en tolérant les écarts fréquents des LLM (lignes vides manquantes autour des
        tableaux, titres et listes)."""
        out, prev = [], ""
        for line in (text or "").splitlines():
            s = line.strip()
            is_table, was_table = s.startswith("|"), prev.startswith("|")
            bullet, num = re.match(r"^[-*] ", s), re.match(r"^\d+\. ", s)
            is_block = (s.startswith("#") or (bullet and not re.match(r"^[-*] ", prev))
                        or (num and not re.match(r"^\d+\. ", prev)))
            if prev and ((is_table and not was_table) or (was_table and not is_table and s) or is_block):
                out.append("")
            out.append(line)
            prev = s
        return md_lib.markdown("\n".join(out), extensions=["tables", "fenced_code", "sane_lists"])

    @staticmethod
    def card(title="", body="", kicker=None, highlight=False, raw=False):
        k = f'<div class="ntl-kicker">{kicker}</div>' if kicker else ""
        t = f'<div class="ntl-title">{title}</div>' if title else ""
        html = f'<div class="ntl-card{" hl" if highlight else ""}">{k}{t}{body}</div>'
        return html if raw else UI.show(html)

    @staticmethod
    def kpis(items):
        """items : liste de dict(label, value, unit, delta)."""
        cells = []
        for it in items:
            d = it.get("delta")
            dh = ""
            if d is not None and not (isinstance(d, float) and math.isnan(d)):
                cls = "ntl-up" if d >= 0 else "ntl-down"
                dh = f'<div class="d {cls}">{"▲" if d >= 0 else "▼"} {abs(d):.1f} %</div>'
            cells.append(f'<div class="ntl-kpi"><div class="v">{it["value"]} <span class="u">{it.get("unit","")}</span></div>'
                         f'<div class="l">{it["label"]}</div>{dh}</div>')
        return f'<div class="ntl-kpis">{"".join(cells)}</div>'

    @staticmethod
    def pills(items, cls=""):
        return "".join(f'<span class="ntl-pill {cls}">{i}</span>' for i in items)

    @staticmethod
    def ok(msg):    UI.show(f'<div class="ntl-ok">✅ {msg}</div>')
    @staticmethod
    def warn(msg):  UI.show(f'<div class="ntl-warn">⚠️ {msg}</div>')
    @staticmethod
    def alert(msg): UI.show(f'<div class="ntl-alert">⛔ {msg}</div>')

ui = UI()
ui.card("Charte graphique chargée",
        ui.kpis([dict(label="Couleur primaire", value="#00A86A"), dict(label="Accent", value="#F5C242"),
                 dict(label="Thème Plotly", value="stg17")]),
        kicker="03 · Identité visuelle")

### Vue d'ensemble : ce que nous allons construire

Un **agent** est un LLM placé dans une boucle : il lit la question, **décide** quel outil appeler, **observe** le résultat, puis recommence jusqu'à pouvoir répondre. Le LLM ne calcule rien lui-même : **tous les chiffres viennent des outils**, qui interrogent Earth Engine. Ce principe est la meilleure protection contre les hallucinations.

In [ ]:
def show_architecture():
    box = ("background:{bg};border:1.5px solid {bd};border-radius:12px;padding:12px 14px;min-width:150px;"
           "flex:1;text-align:center")
    arrow = '<div style="align-self:center;font-size:26px;color:#5E6964;padding:0 4px">⇄</div>'
    def b(title, sub, bg="#fff", bd="#00A86A", icon=""):
        return (f'<div style="{box.format(bg=bg, bd=bd)}"><div style="font-size:26px">{icon}</div>'
                f'<div style="font-weight:700;margin:2px 0">{title}</div><div class="ntl-muted">{sub}</div></div>')
    flow = ('<div style="display:flex;flex-wrap:wrap;gap:6px;align-items:stretch">'
            + b("Question", "en langage naturel<br>+ code ISO3", icon="💬", bd="#D5DED9")
            + arrow + b("LLM (le cerveau)", "Groq · Gemini · Ollama<br>planifie, choisit les outils", "#E8F5EF", icon="🧠")
            + arrow + b("Boîte à outils", "21 fonctions Python<br>schémas JSON auto-générés", icon="🧰", bd="#0E7C86")
            + arrow + b("Données", "Earth Engine : VIIRS, Black Marble,<br>WorldPop, geoBoundaries · Banque mondiale", icon="🛰️", bd="#D49A00")
            + '</div>'
            '<div style="text-align:center;font-size:24px;color:#5E6964;margin:4px 0">⇩</div>'
            '<div style="display:flex;justify-content:center">'
            + b("Réponse structurée", "En bref · Résultats · Interprétation · <b>Limites</b> · Méthode<br>"
                "+ indicateurs clés, graphiques interactifs, cartes, trace des appels", "#fff", "#00704A", "📑")
            + '</div>')
    ui.card("Architecture de l'agent", flow, kicker="Vue d'ensemble")

show_architecture()

## 04 · Le cerveau : connexion au LLM

### Une seule interface pour trois fournisseurs

Groq, Gemini et Ollama exposent tous une **API compatible OpenAI** (`/chat/completions` avec *tool calling*). On utilise donc **un seul client** (`openai`), et seule l'adresse du serveur change. Changer de fournisseur = changer un paramètre : c'est exactement la situation d'achat que vous comparerez à l'atelier (Jour 2, *Choosing your engine*).

| Fournisseur | Où tourne le modèle ? | Forces | Points d'attention | Modèle préféré |
|---|---|---|---|---|
| **Groq** | Cloud (puces LPU) | Très rapide, niveau gratuit | Limites de débit (erreur 429 gérée automatiquement) ; données envoyées à un tiers | `openai/gpt-oss-120b` |
| **Gemini** | Cloud Google | Long contexte, bon raisonnement, niveau gratuit | Données envoyées à un tiers ; quotas par minute | `gemini-2.5-flash` |
| **Ollama** | **Votre machine** | **Souveraineté** : rien ne sort ; gratuit | Lent sans GPU ; petits modèles moins fiables → utilisez `TOOLSET="core"` | `qwen2.5:7b` |

> Les catalogues de modèles évoluent vite et des modèles sont régulièrement retirés. Le client **vérifie automatiquement** que le modèle existe pour votre clé ; sinon il bascule sur le premier modèle disponible d'une liste de préférence (tous capables d'appeler des outils) et vous l'indique. Pour imposer un modèle, renseignez `LLM_MODEL` (section 01) ; `llm.list_models()` affiche la liste exacte.

> 🔐 **Confidentialité** : l'agent n'envoie au LLM que des **agrégats** (statistiques par région), jamais de microdonnées. Pour des données confidentielles, préférez Ollama (exécution locale).

In [ ]:
import openai
from openai import OpenAI

# Les catalogues évoluent vite (des modèles sont régulièrement retirés) : on donne une liste ORDONNÉE de modèles
# préférés sachant appeler des outils. Au démarrage, le client interroge la liste des modèles réellement disponibles
# pour VOTRE clé et prend le premier présent — plus d'erreur « model_not_found ».
PROVIDERS = {
    "groq":   dict(base_url="https://api.groq.com/openai/v1", key_env="GROQ_API_KEY",
                   preferred=["openai/gpt-oss-120b", "moonshotai/kimi-k2-instruct-0905", "moonshotai/kimi-k2-instruct",
                              "qwen/qwen3-32b", "meta-llama/llama-4-maverick-17b-128e-instruct",
                              "meta-llama/llama-4-scout-17b-16e-instruct", "llama-3.3-70b-versatile",
                              "openai/gpt-oss-20b", "llama-3.1-8b-instant"]),
    "gemini": dict(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", key_env="GEMINI_API_KEY",
                   preferred=["gemini-2.5-flash", "gemini-flash-latest", "gemini-2.5-pro", "gemini-2.5-flash-lite",
                              "gemini-2.0-flash"]),
    "ollama": dict(base_url=os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/") + "/v1", key_env=None,
                   preferred=["qwen2.5:7b", "qwen3:8b", "llama3.1:8b", "mistral-nemo", "qwen2.5:14b"]),
}
for _cfg in PROVIDERS.values():
    _cfg["default_model"] = _cfg["preferred"][0]
    _cfg["alternatives"] = _cfg["preferred"][1:4]

# Modèles à écarter lors du choix automatique (audio, sécurité, embeddings…)
_NOT_CHAT = ("whisper", "tts", "guard", "embed", "playai", "orpheus", "compound", "aqa", "imagen", "veo",
             "image", "audio", "live", "safeguard")


def get_secret(name, prompt=True):
    """Cherche un secret : variable d'environnement → Secrets Colab → Secrets Kaggle → fichier .env → saisie masquée."""
    val = os.environ.get(name)
    if val:
        return val
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    elif IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
    if not val and Path(".env").exists():
        for line in Path(".env").read_text().splitlines():
            if line.strip().startswith(f"{name}="):
                val = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not val and prompt:
        from getpass import getpass
        val = getpass(f"🔑 {name} (saisie masquée) : ").strip()
    if val:
        os.environ[name] = val
    return val or None


class LLMClient:
    """Client unique (API compatible OpenAI) pour Groq, Gemini et Ollama, avec reprise automatique sur 429."""

    def __init__(self, provider, model=None, prompt_for_key=True):
        if provider not in PROVIDERS:
            raise ValueError(f"Fournisseur inconnu : {provider}. Choix : {list(PROVIDERS)}")
        cfg = PROVIDERS[provider]
        key = "ollama"
        if cfg["key_env"]:
            key = get_secret(cfg["key_env"], prompt=prompt_for_key)
            if not key:
                raise RuntimeError(f"Clé {cfg['key_env']} introuvable.")
        self.provider = provider
        self.client = OpenAI(base_url=cfg["base_url"], api_key=key, timeout=180, max_retries=2)
        self.model = self._resolve_model(model or cfg["default_model"], cfg["preferred"])

    def _resolve_model(self, wanted, preferred):
        """Vérifie que le modèle existe pour cette clé ; sinon choisit automatiquement le meilleur disponible."""
        try:
            available = self.list_models()
        except Exception:
            return wanted                      # liste indisponible : on tente le modèle demandé
        if not available or wanted in available:
            return wanted
        choice = next((m for m in preferred if m in available), None)
        if choice is None:
            chat = [m for m in available if not any(k in m.lower() for k in _NOT_CHAT)]
            choice = (chat or available)[0]
        print(f"ℹ️ Modèle « {wanted} » indisponible chez {self.provider} → bascule automatique sur « {choice} ».\n"
              f"   Modèles disponibles : {', '.join(available[:15])}{' …' if len(available) > 15 else ''}")
        return choice

    def __repr__(self):
        return f"LLMClient({self.provider}:{self.model})"

    def chat(self, messages, tools=None, temperature=TEMPERATURE):
        kwargs = dict(model=self.model, messages=messages, temperature=temperature)
        if tools:
            kwargs.update(tools=tools, tool_choice="auto")
        for attempt in range(5):
            try:
                return self.client.chat.completions.create(**kwargs)
            except openai.RateLimitError:
                wait = min(60, 6 * 2 ** attempt)
                print(f"⏳ Limite de débit atteinte chez {self.provider} — nouvelle tentative dans {wait} s…")
                time.sleep(wait)
            except openai.APIConnectionError as e:
                if self.provider == "ollama":
                    raise RuntimeError("Ollama injoignable : lancez `ollama serve` (section 04) "
                                       "et vérifiez OLLAMA_HOST.") from e
                if attempt >= 1:
                    raise
                time.sleep(3)
        raise RuntimeError("Échec après plusieurs tentatives (limite de débit).")

    def list_models(self):
        """Modèles réellement accessibles avec votre clé (préfixe « models/ » de Gemini retiré)."""
        return sorted(m.id.removeprefix("models/") for m in self.client.models.list())

    def ping(self):
        t0 = time.time()
        r = self.chat([{"role": "user", "content": "Réponds uniquement par le mot : OK"}], temperature=0)
        return (r.choices[0].message.content or "").strip(), time.time() - t0

### (Optionnel) Ollama sur Colab / Kaggle

Si vous avez choisi `ollama`, la cellule suivante installe et lance le serveur Ollama **dans la machine virtuelle** Colab/Kaggle, puis télécharge le modèle. Activez un **GPU** (Colab : *Exécution › Modifier le type d'exécution* ; Kaggle : *Accelerator*), sinon les réponses seront très lentes. En local, installez Ollama vous-même puis `ollama pull qwen2.5:7b`.
Pour les autres fournisseurs, la cellule ne fait rien.

In [ ]:
import shutil
if LLM_PROVIDER == "ollama":
    _model = LLM_MODEL or PROVIDERS["ollama"]["default_model"]
    if IN_COLAB or IN_KAGGLE:
        if not shutil.which("ollama"):
            print("⬇️ Installation d'Ollama (≈1 min)…")
            subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null 2>&1; "
                           "curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(6)
    if shutil.which("ollama"):
        print(f"⬇️ Téléchargement du modèle {_model} (peut prendre quelques minutes la première fois)…")
        subprocess.run(["ollama", "pull", _model], stdout=subprocess.DEVNULL)
        print("✅ Ollama prêt.")
    else:
        print("ℹ️ Ollama n'est pas installé ici : https://ollama.com/download puis `ollama serve`.")
else:
    print(f"ℹ️ Fournisseur « {LLM_PROVIDER} » : cette cellule est ignorée.")

In [ ]:
try:
    llm = LLMClient(LLM_PROVIDER, LLM_MODEL or None)
    reply, latency = llm.ping()
    ui.card(f"Connecté à {llm.provider.capitalize()}",
            ui.kpis([dict(label="Modèle", value=llm.model),
                     dict(label="Latence d'un aller-retour", value=f"{latency:.2f}", unit="s"),
                     dict(label="Réponse au test", value=reply[:20] or "∅")]),
            kicker="04 · Le cerveau", highlight=True)
except Exception as e:
    llm = None
    alt = ", ".join(PROVIDERS[LLM_PROVIDER]["alternatives"])
    ui.alert(f"Connexion impossible à <b>{LLM_PROVIDER}</b> : <code>{str(e)[:300]}</code><br>"
             f"Vérifiez la clé, le nom du modèle (alternatives : {alt}) ou changez de fournisseur en section 01. "
             "Les sections 05 à 07 (données et outils) fonctionnent sans LLM.")

## 05 · Les données : lumières nocturnes

### Ce que mesurent les satellites la nuit

Le capteur **VIIRS Day/Night Band** (satellites Suomi-NPP depuis 2012, puis NOAA-20/21) mesure la **radiance** nocturne en **nW/cm²/sr** à ≈ 500 m de résolution. Il succède à **DMSP-OLS** (1992–2013), plus grossier et saturé en ville : les deux séries **ne se raccordent pas directement** (rupture de capteur).

| Produit (ID Earth Engine) | Fréquence | Période | Bande utilisée | Usage dans l'agent |
|---|---|---|---|---|
| `NOAA/VIIRS/DNB/ANNUAL_V21` + `ANNUAL_V22` (EOG VNL v2) | Annuelle | 2012/13 → récent | `average_masked` (fond, feux et aurores masqués) | Séries annuelles, statistiques par région |
| `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG` | Mensuelle | 2014 → récent | `avg_rad`, `cf_cvg` (nb. d'observations sans nuage) | Séries mensuelles, anomalies, qualité |
| `NASA/VIIRS/002/VNP46A2` (Black Marble) | Journalière | 2012 → récent | `DNB_BRDF_Corrected_NTL` + drapeau qualité | Impact d'un événement (avant/après) |
| `projects/sat-io/open-datasets/WORLDPOP/pop` (WorldPop **Global2**, R2025A) | Annuelle | **2015–2030** (2026–2030 : projections) | `population` | Population en zone non éclairée, NTL par habitant |
| `WorldPop/GP/100m/pop` (WorldPop GP, v1) | Annuelle | 2000–2014 (secours) | `population` | Années antérieures à 2015 |
| `WM/geoLab/geoBoundaries/600/ADM{0,1,2}` | — | — | `shapeGroup` = code ISO3 | Frontières et régions |

### Les indicateurs calculés

| Indicateur | Définition | Lecture |
|---|---|---|
| **SOL** (*Sum Of Lights*) | Somme des radiances des pixels d'une zone | Proxy de l'activité / de l'électrification totale |
| **Radiance moyenne** | SOL ÷ nombre de pixels | Intensité, comparable entre zones de tailles différentes |
| **Surface éclairée** | Surface des pixels > `LIT_THRESHOLD` | Étalement de l'urbanisation / de l'électrification |
| **Population non éclairée** | Population WorldPop vivant dans des pixels sous le seuil | Proxy (imparfait) du déficit d'accès à l'électricité |

### Les artefacts à connaître (et comment la boîte à outils les traite)

| Artefact | Symptôme | Parade dans l'agent |
|---|---|---|
| **Couverture nuageuse** | Baisse brutale un mois de saison des pluies | `detect_anomalies` croise chaque anomalie avec `coverage_pct` |
| **Saisonnalité** | Cycle annuel régulier | Désaisonnalisation avant détection des anomalies |
| **Torchères (gas flares)** | Points extrêmement brillants hors des villes | `get_quality_report` : percentiles, maximum, alerte pays producteurs |
| **Blooming** | Halo lumineux autour des villes | Seuil d'éclairage ; prudence sur les petites unités ADM2 |
| **Bruit en zone rurale peu éclairée** | Valeurs faibles instables | Seuil `LIT_THRESHOLD` ; produit annuel masqué |
| **Rupture de produit** | Saut de niveau entre deux sources | Une série utilise **une seule source** d'un bout à l'autre (paramètre `source`) |

### Connexion à Earth Engine (ou mode démo)

La logique de `DATA_BACKEND` :

* `gee` → Earth Engine obligatoire (erreur explicite sinon) ;
* `demo` → données **synthétiques** générées localement (apprentissage de la mécanique uniquement) ;
* `auto` → Earth Engine si un projet est configuré et l'authentification réussit, sinon démo.

Les deux « backends » exposent **exactement les mêmes méthodes** : la boîte à outils et l'agent ne voient pas la différence. C'est un bon réflexe d'architecture, qui permet par exemple d'ajouter demain un backend basé sur des fichiers Black Marble téléchargés depuis NASA Earthdata.

In [ ]:
import pycountry


def country_name(iso3):
    c = pycountry.countries.get(alpha_3=str(iso3).strip().upper())
    if c is None:
        raise ValueError(f"Code ISO3 inconnu : « {iso3} ». Exemples valides : RWA, SEN, CIV, KEN.")
    return getattr(c, "common_name", None) or c.name


class DiskCache:
    """Cache disque très simple : évite de relancer des calculs Earth Engine identiques (gain de temps et de quota)."""
    def __init__(self, folder):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)

    def get(self, key_parts, compute):
        key = hashlib.md5(json.dumps(key_parts, sort_keys=True, default=str).encode()).hexdigest()
        path = self.folder / f"{key}.pkl"
        if path.exists():
            try:
                return pickle.loads(path.read_bytes())
            except Exception:
                pass
        value = compute()
        path.write_bytes(pickle.dumps(value))
        return value

    def clear(self):
        for p in self.folder.glob("*.pkl"):
            p.unlink()

CACHE = DiskCache(OUT / "cache")


def _finalize(df):
    """Ajoute les indicateurs dérivés à partir des sommes brutes renvoyées par les backends."""
    df = df.copy()
    if {"sol", "n_px"} <= set(df.columns):
        df["mean_rad"] = df["sol"] / df["n_px"].replace(0, np.nan)
    if {"lit_px", "n_px"} <= set(df.columns):
        df["lit_share"] = 100 * df["lit_px"] / df["n_px"].replace(0, np.nan)
    if {"cov_px", "n_px"} <= set(df.columns):
        df["coverage_pct"] = 100 * df["cov_px"] / df["n_px"].replace(0, np.nan)
    return df.drop(columns=[c for c in ("lit_px", "cov_px") if c in df.columns])


class BaseBackend:
    """Méthodes communes aux deux backends (résolution des sources et des années)."""
    synthetic = False
    PIXEL_KM2 = 0.2146   # surface d'un pixel VIIRS 15" à l'équateur

    @property
    def annual_years(self):
        return sorted({y for ys in self.availability["annual"].values() for y in ys})

    @property
    def monthly_full_years(self):
        months = self.availability["monthly_months"]
        counts = pd.Series([m[:4] for m in months]).value_counts()
        return sorted(int(y) for y, n in counts.items() if n == 12)

    def resolve_source(self, years, source="auto"):
        years = set(int(y) for y in years)
        ann, mon = set(self.annual_years), set(self.monthly_full_years)
        if source == "vnl_annual" or (source == "auto" and years <= ann):
            miss = sorted(years - ann)
            if miss:
                raise ValueError(f"Années absentes du produit annuel VNL : {miss}. Disponibles : {min(ann)}–{max(ann)}. "
                                 "Essayez source='monthly_mean' ou d'autres années.")
            return "vnl_annual"
        miss = sorted(years - mon)
        if miss:
            allowed = sorted(ann | mon)
            raise ValueError(f"Années indisponibles : {miss}. Années utilisables : {allowed[0]}–{allowed[-1]}.")
        return "monthly_mean"

    def source_label(self, src):
        return {"vnl_annual": "VIIRS VNL v2 annuel (average_masked)",
                "monthly_mean": "Moyenne des composites mensuels VIIRS VCMSLCFG"}.get(src, src)

    def default_year(self):
        return max(self.annual_years) if self.annual_years else max(self.monthly_full_years)

In [ ]:
NIGHT_HEX = [c.lstrip("#") for c in NIGHT]


def _ee_initialize(project):
    """Initialise Earth Engine selon l'environnement (compte de service Kaggle, Colab, local)."""
    import ee
    sa_json = get_secret("GEE_SERVICE_ACCOUNT_JSON", prompt=False)
    if sa_json:
        info = json.loads(sa_json)
        creds = ee.ServiceAccountCredentials(info["client_email"], key_data=sa_json)
        ee.Initialize(creds, project=project or info.get("project_id"))
        return ee
    try:
        ee.Initialize(project=project)
    except Exception:
        if IN_COLAB:
            ee.Authenticate()
        else:
            ee.Authenticate(auth_mode="notebook")
        ee.Initialize(project=project)
    return ee


class GEEBackend(BaseBackend):
    name = "Google Earth Engine"
    MONTHLY_ID = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"
    ANNUAL_IDS = ("NOAA/VIIRS/DNB/ANNUAL_V21", "NOAA/VIIRS/DNB/ANNUAL_V22")
    DAILY_ID   = "NASA/VIIRS/002/VNP46A2"
    POP_ID     = "WorldPop/GP/100m/pop"                         # 2000–2020 (catalogue officiel)
    POP_G2_ID  = "projects/sat-io/open-datasets/WORLDPOP/pop"     # WorldPop Global2 R2025A, 2015–2030
    BOUNDS_ID  = "WM/geoLab/geoBoundaries/600/ADM{level}"
    SCALE      = 463.83

    def __init__(self, project):
        self.ee = _ee_initialize(project)
        self.project = project
        self.monthly = self.ee.ImageCollection(self.MONTHLY_ID)
        self._fc, self._n = {}, {}
        today = dt.date.today().isoformat()[:7]          # disponibilité recalculée chaque mois
        self.availability = CACHE.get(["gee-availability", today], self._availability)

    # ---------- disponibilité ----------
    def _availability(self):
        ee = self.ee
        def years_of(cid):
            try:
                ts = ee.ImageCollection(cid).aggregate_array("system:time_start").getInfo()
                return sorted({dt.datetime.fromtimestamp(t / 1000, dt.timezone.utc).year for t in ts})
            except Exception:
                return []
        ann = {cid: years_of(cid) for cid in self.ANNUAL_IDS}
        ts = self.monthly.aggregate_array("system:time_start").getInfo()
        months = sorted({dt.datetime.fromtimestamp(t / 1000, dt.timezone.utc).strftime("%Y-%m") for t in ts})
        return {"annual": ann, "monthly_months": months}

    # ---------- frontières ----------
    def _admin_fc(self, iso3, level):
        ee, key = self.ee, (iso3, level)
        if key in self._fc:
            return self._fc[key]
        fc = ee.FeatureCollection(self.BOUNDS_ID.format(level=level)).filter(ee.Filter.eq("shapeGroup", iso3))
        try:
            n = fc.size().getInfo()
        except Exception:
            n = 0
        if n == 0:   # repli : FAO GAUL 2015 (filtré par nom de pays)
            gaul = ee.FeatureCollection(f"FAO/GAUL/2015/level{level}").filter(
                ee.Filter.eq("ADM0_NAME", pycountry.countries.get(alpha_3=iso3).name))
            fc = gaul.map(lambda f: ee.Feature(f.geometry(), {
                "shapeID": ee.Number(f.get(f"ADM{level}_CODE")).format("%d"),
                "shapeName": f.get(f"ADM{level}_NAME")}))
            n = fc.size().getInfo()
            if n == 0:
                raise ValueError(f"Aucune unité ADM{level} trouvée pour {iso3} (geoBoundaries et GAUL).")
        self._fc[key] = fc.select(["shapeID", "shapeName"])
        self._n[key] = n
        return self._fc[key]

    def admin_units(self, iso3, level):
        def compute():
            fc = self._admin_fc(iso3, level)
            info = fc.map(lambda f: self.ee.Feature(None, {
                "unit_id": f.get("shapeID"), "unit": f.get("shapeName"),
                "area_km2": f.geometry().area(1000).divide(1e6)})).getInfo()
            return pd.DataFrame([x["properties"] for x in info["features"]])
        return CACHE.get(["gee-units", iso3, level], compute).copy()

    def geojson(self, iso3, level):
        def compute():
            fc = self._admin_fc(iso3, level)
            tol = 2000 if level == 0 else 800
            simp = fc.map(lambda f: self.ee.Feature(f.simplify(maxError=tol).geometry(),
                                                     {"unit_id": f.get("shapeID"), "unit": f.get("shapeName")}))
            return simp.getInfo()
        return CACHE.get(["gee-geojson", iso3, level], compute)

    # ---------- images ----------
    def _annual_rad(self, year, src):
        ee = self.ee
        if src == "vnl_annual":
            cid = next(c for c, ys in self.availability["annual"].items() if year in ys)
            img = ee.ImageCollection(cid).filterDate(f"{year}-01-01", f"{year + 1}-01-01").first()
            rad = ee.Image(img).select("average_masked").unmask(0).max(0)
        else:
            col = self.monthly.filterDate(f"{year}-01-01", f"{year + 1}-01-01").select("avg_rad")
            rad = col.map(lambda i: i.max(0)).mean().unmask(0)
            rad = rad.where(rad.lt(LIT_THRESHOLD), 0)
        return rad.rename("rad").set("year", year)

    def _stats_image(self, rad):
        ee = self.ee
        area = ee.Image.pixelArea().divide(1e6)
        lit = rad.gt(LIT_THRESHOLD)
        return ee.Image.cat([rad.rename("sol"), lit.multiply(area).rename("lit_km2"), area.rename("area_km2"),
                             lit.rename("lit_px"), ee.Image.constant(1).rename("n_px")])

    # ---------- séries nationales ----------
    def national_series(self, iso3, start_year, end_year, frequency="annual", source="auto"):
        return CACHE.get(["gee-nat", iso3, start_year, end_year, frequency, source, LIT_THRESHOLD],
                         lambda: self._national_series(iso3, start_year, end_year, frequency, source)).copy()

    def _national_series(self, iso3, start_year, end_year, frequency, source):
        ee = self.ee
        geom = self._admin_fc(iso3, 0).geometry()
        red_kw = dict(reducer=ee.Reducer.sum(), geometry=geom, scale=self.SCALE, maxPixels=1e13, tileScale=4)
        if frequency == "annual":
            years = list(range(start_year, end_year + 1))
            src = self.resolve_source(years, source)
            col = ee.ImageCollection.fromImages([self._annual_rad(y, src).set("period", str(y)) for y in years])
            def red(img):
                d = self._stats_image(ee.Image(img).select("rad")).reduceRegion(**red_kw)
                return ee.Feature(None, d).set("period", img.get("period"))
            label = self.source_label(src)
        else:
            col = self.monthly.filterDate(f"{start_year}-01-01", f"{end_year + 1}-01-01")
            thr = LIT_THRESHOLD
            def red(img):
                rad = img.select("avg_rad").max(0).unmask(0)
                rad = rad.where(rad.lt(thr), 0).rename("rad")
                st = self._stats_image(rad).addBands(img.select("cf_cvg").gt(0).unmask(0).rename("cov_px"))
                return ee.Feature(None, st.reduceRegion(**red_kw)).set("period", img.date().format("YYYY-MM"))
            label = "VIIRS VCMSLCFG mensuel (avg_rad, seuil appliqué)"
        rows = [f["properties"] for f in ee.FeatureCollection(col.map(red)).getInfo()["features"]]
        df = _finalize(pd.DataFrame(rows)).sort_values("period").reset_index(drop=True)
        df["source"] = label
        return df

    # ---------- statistiques zonales ----------
    def zonal_stats(self, iso3, level, years, source="auto"):
        years = sorted(int(y) for y in years)
        return CACHE.get(["gee-zonal", iso3, level, years, source, LIT_THRESHOLD],
                         lambda: self._zonal_stats(iso3, level, years, source)).copy()

    def _zonal_stats(self, iso3, level, years, source):
        ee = self.ee
        src = self.resolve_source(years, source)
        fc = self._admin_fc(iso3, level)
        chunk = max(1, 4000 // max(1, self._n[(iso3, level)]))
        rows = []
        for i in range(0, len(years), chunk):
            imgs = [self._annual_rad(y, src) for y in years[i:i + chunk]]
            def per_img(img):
                img = ee.Image(img)
                y = img.get("year")
                red = self._stats_image(img.select("rad")).reduceRegions(
                    collection=fc, reducer=ee.Reducer.sum(), scale=self.SCALE, tileScale=4)
                return red.map(lambda f: ee.Feature(None, f.toDictionary()).set("year", y))
            res = ee.FeatureCollection(ee.ImageCollection.fromImages(imgs).map(per_img)).flatten().getInfo()
            rows += [f["properties"] for f in res["features"]]
        df = pd.DataFrame(rows).rename(columns={"shapeID": "unit_id", "shapeName": "unit"})
        df = _finalize(df)
        df["year"] = df["year"].astype(int)
        df["source"] = self.source_label(src)
        return df

    # ---------- population ----------
    def population_stats(self, iso3, level, year, source="auto"):
        return CACHE.get(["gee-pop-g2", iso3, level, year, source, LIT_THRESHOLD],
                         lambda: self._population_stats(iso3, level, year, source)).copy()

    def _worldpop(self, iso3, year):
        """WorldPop Global2 (2015–2030) d'abord, WorldPop GP v1 (2000–2020) en secours.
        Renvoie (image population, année utilisée, libellé de la source)."""
        ee = self.ee
        geom = self._admin_fc(iso3, 0).geometry()
        y = int(min(max(year, 2015), 2030))
        if year >= 2015:
            try:
                col = (ee.ImageCollection(self.POP_G2_ID).filterDate(f"{y}-01-01", f"{y + 1}-01-01")
                       .filterBounds(geom))
                if col.size().getInfo() > 0:
                    proj = col.first().select(0).projection()
                    img = col.mosaic().select(0).rename("population").setDefaultProjection(proj)
                    return img, y, "WorldPop Global2 R2025A" + (" (projection)" if y > 2025 else "")
            except Exception:
                pass
        y = int(min(max(year, 2000), 2020))
        col = ee.ImageCollection(self.POP_ID).filter(ee.Filter.eq("country", iso3)).filter(ee.Filter.eq("year", y))
        if col.size().getInfo() == 0:
            raise ValueError(f"WorldPop indisponible pour {iso3} en {year}.")
        return col.first().select("population"), y, "WorldPop GP v1"

    def _population_stats(self, iso3, level, year, source):
        ee = self.ee
        rad = self._annual_rad(year, self.resolve_source([year], source))
        pop, pop_year, pop_source = self._worldpop(iso3, year)
        area = float(self.admin_units(iso3, 0)["area_km2"].sum())
        if area < 300_000:                       # résolution native (100 m)
            pop_img, scale = pop, 100
        else:                                    # grands pays : densité × surface (évite les délais de calcul)
            density = pop.divide(ee.Image.pixelArea().reproject(pop.projection()))
            pop_img, scale = density.multiply(ee.Image.pixelArea()), 250
        img = ee.Image.cat([pop_img.rename("pop"), pop_img.multiply(rad.lt(LIT_THRESHOLD)).rename("pop_unlit")])
        res = img.reduceRegions(collection=self._admin_fc(iso3, level), reducer=ee.Reducer.sum(),
                                scale=scale, tileScale=16).map(lambda f: ee.Feature(None, f.toDictionary())).getInfo()
        df = pd.DataFrame([f["properties"] for f in res["features"]]).rename(
            columns={"shapeID": "unit_id", "shapeName": "unit"})
        df["pop_year"] = pop_year
        df["pop_source"] = pop_source
        return df

    # ---------- événement (Black Marble journalier) ----------
    def event_impact(self, iso3, level, event_date, days_before=30, days_after=30):
        return CACHE.get(["gee-event", iso3, level, event_date, days_before, days_after],
                         lambda: self._event_impact(iso3, level, event_date, days_before, days_after)).copy()

    def _event_impact(self, iso3, level, event_date, days_before, days_after):
        ee = self.ee
        d = ee.Date(event_date)
        col = ee.ImageCollection(self.DAILY_ID)
        def clean(img):
            q = img.select("Mandatory_Quality_Flag")
            return img.select("DNB_BRDF_Corrected_NTL").updateMask(q.lte(1)).rename("rad")
        before = col.filterDate(d.advance(-days_before, "day"), d).map(clean)
        after = col.filterDate(d, d.advance(days_after, "day")).map(clean)
        n_b, n_a = before.size().getInfo(), after.size().getInfo()
        if n_b == 0 or n_a == 0:
            raise ValueError(f"Pas d'images Black Marble sur la fenêtre (avant : {n_b}, après : {n_a}). "
                             "La date est peut-être trop récente ou antérieure à 2012.")
        img = ee.Image.cat([before.median().unmask(0).rename("sol_before"), after.median().unmask(0).rename("sol_after"),
                            before.count().gt(0).unmask(0).rename("cov_b"), after.count().gt(0).unmask(0).rename("cov_a"),
                            ee.Image.constant(1).rename("n_px")])
        res = img.reduceRegions(collection=self._admin_fc(iso3, level), reducer=ee.Reducer.sum(),
                                scale=500, tileScale=4).map(lambda f: ee.Feature(None, f.toDictionary())).getInfo()
        df = pd.DataFrame([f["properties"] for f in res["features"]]).rename(
            columns={"shapeID": "unit_id", "shapeName": "unit"})
        df["coverage_before_pct"] = 100 * df.pop("cov_b") / df["n_px"]
        df["coverage_after_pct"] = 100 * df.pop("cov_a") / df["n_px"]
        df["n_images_before"], df["n_images_after"] = n_b, n_a
        return df.drop(columns="n_px")

    # ---------- qualité ----------
    def quality_report(self, iso3, year, source="auto"):
        return CACHE.get(["gee-quality", iso3, year, source, LIT_THRESHOLD],
                         lambda: self._quality_report(iso3, year, source))

    def _quality_report(self, iso3, year, source):
        ee = self.ee
        geom = self._admin_fc(iso3, 0).geometry()
        col = self.monthly.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        def per(img):
            cov = img.select("cf_cvg")
            d = ee.Image.cat([cov.gt(0).rename("covered"), cov.rename("cf_cvg")]).reduceRegion(
                ee.Reducer.mean(), geom, 2000, maxPixels=1e13, bestEffort=True)
            return ee.Feature(None, d).set("period", img.date().format("YYYY-MM"))
        cov = pd.DataFrame([f["properties"] for f in ee.FeatureCollection(col.map(per)).getInfo()["features"]])
        cov = cov.rename(columns={"covered": "coverage_share", "cf_cvg": "mean_cloudfree_obs"})
        cov["coverage_pct"] = 100 * cov.pop("coverage_share")
        rad = self._annual_rad(year, self.resolve_source([year], source))
        pct = rad.updateMask(rad.gt(LIT_THRESHOLD)).reduceRegion(
            ee.Reducer.percentile([50, 90, 99]).combine(ee.Reducer.max(), "", True),
            geom, self.SCALE, maxPixels=1e13, bestEffort=True).getInfo()
        return {"monthly_coverage": cov.sort_values("period"),
                "percentiles": {k.replace("rad_", ""): v for k, v in pct.items()}}

    # ---------- image ----------
    def thumbnail(self, iso3, year, source="auto"):
        def compute():
            geom = self._admin_fc(iso3, 0).geometry()
            rad = self._annual_rad(year, self.resolve_source([year], source))
            vis = rad.add(0.1).log10().clip(geom).visualize(min=-1, max=2, palette=NIGHT_HEX)
            url = vis.getThumbURL({"region": geom.bounds(), "dimensions": 1000, "format": "png"})
            return requests.get(url, timeout=180).content
        return CACHE.get(["gee-thumb", iso3, year, source], compute)

In [ ]:
from shapely.geometry import shape, box, mapping
import shapely


class DemoBackend(BaseBackend):
    """Données SYNTHÉTIQUES, déterministes pour un code ISO donné. Frontières réelles (geoBoundaries) si Internet
    le permet, sinon grille fictive. Sert uniquement à apprendre la mécanique de l'agent — ne jamais citer ces chiffres."""
    name = "Mode démo (données synthétiques)"
    synthetic = True

    def __init__(self):
        months = [f"{y}-{m:02d}" for y in range(2014, 2026) for m in range(1, 13)][:138]   # 2014-01 → 2025-06
        self.availability = {"annual": {"DEMO_ANNUAL": list(range(2014, 2025))}, "monthly_months": months}
        self._units = {}

    # ---------- frontières ----------
    def _fetch_geoboundaries(self, iso3, level):
        try:
            meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/{iso3}/ADM{level}/", timeout=15).json()
            gj = requests.get(meta["simplifiedGeometryGeoJSON"], timeout=60).json()
            feats = []
            for i, f in enumerate(gj["features"]):
                p = f.get("properties", {})
                feats.append({"type": "Feature", "geometry": f["geometry"],
                              "properties": {"unit_id": p.get("shapeID", f"{iso3}-{level}-{i}"),
                                             "unit": p.get("shapeName", f"Unité {i + 1}")}})
            return {"type": "FeatureCollection", "features": feats}
        except Exception:
            return None

    def _synthetic_grid(self, iso3, level):
        rng = np.random.default_rng(zlib.crc32(f"{iso3}-grid".encode()))
        nx, ny = (3, 2) if level == 0 else ((4, 3) if level == 1 else (8, 6))
        if level == 0:
            nx = ny = 1
        x0, y0, w, h = 28.8, -2.9, 2.2, 1.9
        feats = []
        for j in range(ny):
            for i in range(nx):
                k = j * nx + i
                g = box(x0 + i * w / nx, y0 + j * h / ny, x0 + (i + 1) * w / nx, y0 + (j + 1) * h / ny)
                name = country_name(iso3) if level == 0 else f"Région {chr(65 + k)}" if level == 1 else f"District {k + 1:02d}"
                feats.append({"type": "Feature", "geometry": mapping(g),
                              "properties": {"unit_id": f"{iso3}-ADM{level}-{k + 1:03d}", "unit": name}})
        return {"type": "FeatureCollection", "features": feats}

    def geojson(self, iso3, level):
        return CACHE.get(["demo-geojson", iso3, level],
                         lambda: self._fetch_geoboundaries(iso3, level) or self._synthetic_grid(iso3, level))

    def _unit_table(self, iso3, level):
        key = (iso3, level)
        if key in self._units:
            return self._units[key]
        gj = self.geojson(iso3, level)
        rows = []
        for f in gj["features"]:
            g = shape(f["geometry"])
            c = g.centroid
            area = g.area * 111.32 ** 2 * math.cos(math.radians(c.y))
            rows.append(dict(unit_id=f["properties"]["unit_id"], unit=f["properties"]["unit"],
                             area_km2=area, lon=c.x, lat=c.y))
        df = pd.DataFrame(rows)
        rng = np.random.default_rng(zlib.crc32(f"{iso3}-{level}".encode()))
        n = len(df)
        df["base_rad"] = rng.lognormal(mean=np.log(0.35), sigma=0.8, size=n)          # radiance moyenne 2014
        capital = int(rng.integers(n)) if n > 1 else 0
        df.loc[capital, "base_rad"] = df["base_rad"].max() * 6 + 3
        df["growth"] = np.clip(rng.normal(0.055, 0.035, n), -0.03, 0.16)
        df["pop_density"] = rng.lognormal(np.log(180), 0.6, n) * (1 + 3 * df["base_rad"] / df["base_rad"].max())
        df["is_capital"] = False
        df.loc[capital, "is_capital"] = True
        self._units[key] = df
        return df

    @staticmethod
    def _noise(*parts, sigma=0.03):
        return float(np.random.default_rng(zlib.crc32("|".join(map(str, parts)).encode())).normal(1, sigma))

    def _unit_rad(self, iso3, u, year, month=None):
        rad = u.base_rad * (1 + u.growth) ** (year - 2014) * self._noise(iso3, u.unit_id, year)
        if year == 2020:
            rad *= 0.95                                   # choc synthétique « 2020 »
        if month is not None:
            rad *= 1 + 0.06 * math.cos(2 * math.pi * (month - 12) / 12)   # saisonnalité : décembre plus lumineux
            rad *= self._noise(iso3, u.unit_id, year, month, sigma=0.04)
        return rad

    def _coverage(self, iso3, year, month):
        rainy = month in (4, 5, 11)
        cov = (0.62 if rainy else 0.95) * self._noise(iso3, "cov", year, month, sigma=0.05)
        return float(np.clip(cov, 0.3, 1.0))

    def _row(self, iso3, u, rad, cov=None):
        n_px = u.area_km2 / self.PIXEL_KM2
        lit_share = float(np.clip(0.02 + 0.9 * (1 - math.exp(-rad / 1.2)), 0, 0.98))
        r = dict(sol=rad * n_px, n_px=n_px, area_km2=u.area_km2, lit_km2=u.area_km2 * lit_share, lit_px=n_px * lit_share)
        if cov is not None:
            r["cov_px"] = n_px * cov
        return r

    # ---------- API commune ----------
    def admin_units(self, iso3, level):
        return self._unit_table(iso3, level)[["unit_id", "unit", "area_km2"]].copy()

    def national_series(self, iso3, start_year, end_year, frequency="annual", source="auto"):
        units = self._unit_table(iso3, 1)
        rows = []
        if frequency == "annual":
            src = self.resolve_source(range(start_year, end_year + 1), source)
            for y in range(start_year, end_year + 1):
                parts = [self._row(iso3, u, self._unit_rad(iso3, u, y)) for u in units.itertuples()]
                rows.append({"period": str(y), **pd.DataFrame(parts).sum().to_dict()})
            label = self.source_label(src)
        else:
            outage = (2023, 3)                              # vraie baisse synthétique (bonne couverture)
            for m_str in self.availability["monthly_months"]:
                y, m = int(m_str[:4]), int(m_str[5:])
                if not start_year <= y <= end_year:
                    continue
                cov = self._coverage(iso3, y, m)
                factor = (0.72 if (y, m) == outage else 1.0) * (0.8 + 0.2 * cov)   # nuages → radiance sous-estimée
                parts = [self._row(iso3, u, self._unit_rad(iso3, u, y, m) * factor, cov) for u in units.itertuples()]
                rows.append({"period": m_str, **pd.DataFrame(parts).sum().to_dict()})
            label = "Mensuel synthétique"
        df = _finalize(pd.DataFrame(rows).drop(columns=["area_km2"]))
        df["source"] = label + " — DÉMO"
        return df

    def zonal_stats(self, iso3, level, years, source="auto"):
        src = self.resolve_source(years, source)
        units = self._unit_table(iso3, level)
        rows = []
        for y in sorted(int(v) for v in years):
            for u in units.itertuples():
                rows.append(dict(year=y, unit_id=u.unit_id, unit=u.unit, **self._row(iso3, u, self._unit_rad(iso3, u, y))))
        df = _finalize(pd.DataFrame(rows))
        df["source"] = self.source_label(src) + " — DÉMO"
        return df

    def population_stats(self, iso3, level, year, source="auto"):
        self.resolve_source([year], source)
        pop_year = int(min(max(year, 2000), 2030))
        rows = []
        for u in self._unit_table(iso3, level).itertuples():
            pop = u.area_km2 * u.pop_density * (1.026 ** (pop_year - 2014))
            rad = self._unit_rad(iso3, u, year)
            unlit_share = float(np.clip(0.92 * math.exp(-rad / 0.9), 0.02, 0.97))
            rows.append(dict(unit_id=u.unit_id, unit=u.unit, pop=pop, pop_unlit=pop * unlit_share))
        df = pd.DataFrame(rows)
        df["pop_year"] = pop_year
        df["pop_source"] = "WorldPop (DÉMO)"
        return df

    def event_impact(self, iso3, level, event_date, days_before=30, days_after=30):
        d = pd.Timestamp(event_date)
        units = self._unit_table(iso3, level)
        hit = int(np.random.default_rng(zlib.crc32(f"{iso3}{event_date}".encode())).integers(len(units)))
        rows = []
        for i, u in enumerate(units.itertuples()):
            before = self._row(iso3, u, self._unit_rad(iso3, u, d.year, d.month))["sol"]
            change = -0.42 if i == hit else np.random.default_rng(zlib.crc32(f"{u.unit_id}{event_date}".encode())).normal(-0.02, 0.04)
            rows.append(dict(unit_id=u.unit_id, unit=u.unit, sol_before=before, sol_after=before * (1 + change),
                             coverage_before_pct=88.0, coverage_after_pct=84.0,
                             n_images_before=days_before, n_images_after=days_after))
        return pd.DataFrame(rows)

    def quality_report(self, iso3, year, source="auto"):
        self.resolve_source([year], source)
        cov = pd.DataFrame([dict(period=f"{year}-{m:02d}", coverage_pct=100 * self._coverage(iso3, year, m),
                                 mean_cloudfree_obs=round(12 * self._coverage(iso3, year, m), 1)) for m in range(1, 13)])
        units = self._unit_table(iso3, 1)
        top = float(units.base_rad.max() * (1 + units.growth.mean()) ** (year - 2014))
        return {"monthly_coverage": cov, "percentiles": {"p50": 1.4, "p90": 6.8, "p99": round(top * 2.5, 1),
                                                          "max": round(top * 6, 1)}}

    def thumbnail(self, iso3, year, source="auto"):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.colors import LinearSegmentedColormap
        units = self._unit_table(iso3, 1)
        gj = self.geojson(iso3, 0)
        country = shapely.unary_union([shape(f["geometry"]) for f in gj["features"]])
        minx, miny, maxx, maxy = country.bounds
        nx = 700
        ny = max(50, int(nx * (maxy - miny) / (maxx - minx)))
        xs, ys = np.meshgrid(np.linspace(minx, maxx, nx), np.linspace(maxy, miny, ny))
        field = np.full(xs.shape, 0.05)
        rng = np.random.default_rng(zlib.crc32(f"{iso3}-thumb".encode()))
        span = max(maxx - minx, maxy - miny)
        for u in units.itertuples():
            amp = self._unit_rad(iso3, u, year) * 8
            sig = span * (0.012 + 0.02 * u.is_capital)
            field += amp * np.exp(-((xs - u.lon) ** 2 + (ys - u.lat) ** 2) / (2 * sig ** 2))
            for _ in range(12):                        # villes secondaires / villages
                cx, cy = u.lon + rng.normal(0, span * 0.06), u.lat + rng.normal(0, span * 0.06)
                field += amp * 0.08 * rng.random() * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * (span * 0.004) ** 2))
        inside = shapely.contains_xy(country, xs, ys)
        img = np.log10(field + 0.1)
        img = np.where(inside, img, np.nan)
        cmap = LinearSegmentedColormap.from_list("night", NIGHT)
        cmap.set_bad(alpha=0)
        fig, ax = plt.subplots(figsize=(8, 8 * ny / nx), dpi=120)
        ax.imshow(img, cmap=cmap, vmin=-1, vmax=2, interpolation="bilinear")
        ax.axis("off")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0, transparent=True)
        plt.close(fig)
        return buf.getvalue()

In [ ]:
def connect_backend(mode=DATA_BACKEND, project=GEE_PROJECT):
    project = (project or get_secret("GEE_PROJECT", prompt=False) or "").strip()
    if mode in ("gee", "auto"):
        if not project and mode == "auto":
            ui.warn("Aucun projet Earth Engine (<code>GEE_PROJECT</code>) : bascule en <b>mode démo</b> (données synthétiques).")
            return DemoBackend()
        try:
            return GEEBackend(project)
        except Exception as e:
            if mode == "gee":
                raise
            ui.warn(f"Earth Engine indisponible (<code>{str(e)[:200]}</code>) : bascule en <b>mode démo</b>.")
    return DemoBackend()


country_name(COUNTRY_ISO3)            # valide le code ISO3 dès maintenant
backend = connect_backend()
ann = backend.annual_years
mon = backend.availability["monthly_months"]
body = ui.kpis([
    dict(label="Pays", value=COUNTRY_ISO3, unit=country_name(COUNTRY_ISO3)),
    dict(label="Annuel disponible", value=f"{min(ann)}–{max(ann)}" if ann else "—"),
    dict(label="Mensuel disponible", value=f"{mon[0]} → {mon[-1]}"),
    dict(label="Seuil « éclairé »", value=LIT_THRESHOLD, unit="nW/cm²/sr"),
])
if backend.synthetic:
    body += ('<div class="ntl-alert"><b>MODE DÉMO — DONNÉES SYNTHÉTIQUES.</b> Parfait pour apprendre la mécanique de '
             "l'agent ; les valeurs ne décrivent pas la réalité et ne doivent jamais être citées. "
             "Renseignez <code>GEE_PROJECT</code> pour les vraies données.</div>")
ui.card(backend.name, body, kicker="05 · Source de données", highlight=not backend.synthetic)

## 06 · La boîte à outils de l'agent

C'est le cœur du travail. Un LLM ne peut utiliser un outil que s'il **comprend à quoi il sert** et **comment l'appeler**. Cinq principes guident la conception :

1. **La signature Python *est* le contrat.** Le décorateur `@tool` lit les types (`int`, `Literal[...]`, `Optional`) et la docstring pour générer automatiquement le **schéma JSON** envoyé au LLM. Une bonne docstring = un agent qui choisit bien.
2. **Le LLM voit des résumés, Python garde les données.** Chaque outil de calcul range son tableau complet dans la session sous un identifiant (`ds03_admin_stats`) et ne renvoie au LLM qu'un résumé compact. Les outils graphiques réutilisent cet identifiant : le contexte reste court, et les graphiques portent sur les données exactes.
3. **Les erreurs sont des informations.** Une exception est renvoyée au LLM sous forme de message clair (« années disponibles : 2014–2024 ») pour qu'il **corrige lui-même** ses arguments.
4. **Des valeurs par défaut sûres.** Années, niveau administratif, pays : tout a une valeur par défaut raisonnable, ce qui limite les erreurs des petits modèles.
5. **La prudence est intégrée.** Les outils signalent eux-mêmes les limites (couverture nuageuse, rupture de source, petit échantillon, données synthétiques).

In [ ]:
TOOLS = {}                      # nom -> dict(fn, family, core, schema)


def _py_to_schema(ann):
    origin = get_origin(ann)
    if origin is Union:                                        # Optional[X]
        args = [a for a in get_args(ann) if a is not type(None)]
        return _py_to_schema(args[0])
    if origin is Literal:
        vals = list(get_args(ann))
        return {"type": "string", "enum": [str(v) for v in vals]}
    return {"type": {int: "integer", float: "number", bool: "boolean", str: "string"}.get(ann, "string")}


def _parse_doc(doc):
    doc = inspect.cleandoc(doc or "")
    desc, _, args_part = doc.partition("Args:")
    params = {}
    for line in args_part.splitlines():
        m = re.match(r"\s*(\w+)\s*:\s*(.+)", line)
        if m:
            params[m.group(1)] = m.group(2).strip()
    return " ".join(desc.split()), params


def tool(family, core=False):
    """Décorateur : enregistre une fonction Python comme outil et génère son schéma JSON."""
    def deco(fn):
        desc, pdocs = _parse_doc(fn.__doc__)
        props, required = {}, []
        for name, p in inspect.signature(fn).parameters.items():
            sch = _py_to_schema(p.annotation)
            d = pdocs.get(name, "")
            if p.default is inspect.Parameter.empty:
                required.append(name)
            elif p.default is not None:
                d += f" (défaut : {p.default})"
            sch["description"] = d
            props[name] = sch
        TOOLS[fn.__name__] = dict(fn=fn, family=family, core=core, schema={
            "type": "function",
            "function": {"name": fn.__name__, "description": desc,
                         "parameters": {"type": "object", "properties": props, "required": required}}})
        return fn
    return deco


class Session:
    """Mémoire de travail : jeux de données, figures, indicateurs clés et historique des tours."""
    def __init__(self):
        self.datasets, self.figures, self.turns = {}, {}, []
        self.kpis, self.turn_id = [], 0

    def new_turn(self):
        self.turn_id += 1
        self.kpis = []

    def add_dataset(self, df, kind, title, **meta):
        ds_id = f"ds{len(self.datasets) + 1:02d}_{kind}"
        self.datasets[ds_id] = dict(df=df.reset_index(drop=True), kind=kind, title=title, meta=meta, turn=self.turn_id)
        return ds_id

    def get(self, ds_id):
        if ds_id not in self.datasets:
            raise KeyError(f"Jeu de données inconnu : {ds_id}. Disponibles : {list(self.datasets)[-8:]}")
        return self.datasets[ds_id]

    def add_figure(self, obj, title, kind="plotly"):
        fig_id = f"fig{len(self.figures) + 1:02d}"
        self.figures[fig_id] = dict(obj=obj, title=title, kind=kind, turn=self.turn_id)
        if kind == "plotly":
            try:
                obj.write_html(OUT / "figures" / f"{fig_id}.html", include_plotlyjs="cdn")
            except Exception:
                pass
        else:
            (OUT / "figures" / f"{fig_id}.png").write_bytes(obj)
        return fig_id

    def add_kpi(self, label, value, unit="", delta=None):
        self.kpis.append(dict(label=label, value=value, unit=unit, delta=delta))

S = Session()

# ---------- utilitaires partagés ----------
LABELS = {
    "sol": "Somme des radiances (SOL)", "mean_rad": "Radiance moyenne (nW/cm²/sr)",
    "lit_km2": "Surface éclairée (km²)", "lit_share": "Part de surface éclairée (%)",
    "sol_share": "Part du SOL national (%)", "sol_density": "SOL par km²",
    "pct_change": "Variation (%)", "cagr_pct": "Taux de croissance annuel moyen (%)",
    "pop": "Population", "pop_unlit": "Population en zone non éclairée",
    "pop_unlit_share": "Population en zone non éclairée (%)", "sol_per_1000": "SOL pour 1 000 habitants",
    "coverage_pct": "Couverture sans nuage (%)", "indicator": "Indicateur", "sol_before": "SOL avant",
    "sol_after": "SOL après", "lit_km2_change": "Variation de la surface éclairée (km²)",
    "sol_per_capita_1000": "SOL pour 1 000 habitants", "electricity_access": "Accès à l'électricité (%)",
}


def _iso(iso3=None):
    iso = (iso3 or COUNTRY_ISO3).strip().upper()
    country_name(iso)
    return iso


def _level(level):
    level = int(level)
    if level not in (1, 2):
        raise ValueError("Le niveau administratif doit être 1 (régions/provinces) ou 2 (districts).")
    return level


def _years(start_year=None, end_year=None, default_span=10):
    last = backend.default_year()
    end = int(end_year) if end_year else last
    start = int(start_year) if start_year else max(end - default_span + 1, min(backend.annual_years + backend.monthly_full_years))
    if start > end:
        start, end = end, start
    return start, end


def _records(df, n=12, cols=None):
    d = df[cols] if cols else df
    d = d.head(n).copy()
    for c in d.select_dtypes("number").columns:
        d[c] = d[c].map(lambda v: None if pd.isna(v) else (round(float(v), 2) if abs(v) < 1e5 else round(float(v))))
    return d.to_dict("records")


def _cagr(v0, v1, n_years):
    if v0 is None or v0 <= 0 or n_years <= 0:
        return None
    return 100 * ((v1 / v0) ** (1 / n_years) - 1)


def _trend(x, y):
    from scipy import stats
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3:
        return {}
    r = stats.linregress(x, y)
    return dict(slope=r.slope, slope_pct_of_mean=100 * r.slope / np.mean(y), r2=r.rvalue ** 2, p_value=r.pvalue)


def _gini(v):
    v = np.sort(np.asarray(v, float))
    v = v[v >= 0]
    if len(v) == 0 or v.sum() == 0:
        return None
    n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum()))


def _match_units(names, candidates):
    """Rapproche des noms saisis par l'utilisateur des noms officiels (tolère accents et fautes)."""
    out = []
    for n in names:
        m = difflib.get_close_matches(n.strip(), list(candidates), n=1, cutoff=0.5)
        if m:
            out.append(m[0])
    return out


def _base_notes():
    notes = []
    if backend.synthetic:
        notes.append("DONNÉES SYNTHÉTIQUES (mode démo) : ne pas citer ces chiffres comme réels.")
    return notes

### 6.1 · Découverte : cadrer l'analyse

In [ ]:
@tool("Découverte", core=True)
def get_country_profile(iso3: Optional[str] = None) -> dict:
    """Profil du pays : nom, nombre d'unités administratives ADM1 et ADM2, superficie, source de données et années
    disponibles. À appeler en premier pour cadrer une analyse ou connaître les années utilisables.

    Args:
        iso3: Code ISO 3166-1 alpha-3 du pays (ex. RWA, SEN). Par défaut : le pays de la session.
    """
    iso = _iso(iso3)
    adm1 = backend.admin_units(iso, 1)
    try:
        n_adm2 = len(backend.admin_units(iso, 2))
    except Exception:
        n_adm2 = None
    area = float(backend.admin_units(iso, 0)["area_km2"].sum())
    S.add_kpi("Superficie", fmt(area), "km²")
    S.add_kpi("Unités ADM1", len(adm1))
    return {"iso3": iso, "country": country_name(iso), "area_km2": round(area), "n_adm1": len(adm1),
            "n_adm2": n_adm2, "adm1_examples": adm1["unit"].head(8).tolist(),
            "data_backend": backend.name, "annual_years": f"{min(backend.annual_years)}–{max(backend.annual_years)}",
            "monthly_range": f"{backend.availability['monthly_months'][0]} → {backend.availability['monthly_months'][-1]}",
            "lit_threshold_nw": LIT_THRESHOLD, "notes": _base_notes()}


@tool("Découverte")
def get_data_availability() -> dict:
    """Décrit les produits de données disponibles (annuel, mensuel, journalier, population), leurs périodes, les
    définitions des indicateurs (SOL, radiance moyenne, surface éclairée) et les principales limites."""
    return {"products": {
                "annual": {"years": backend.annual_years, "source": backend.source_label("vnl_annual")},
                "monthly": {"first": backend.availability["monthly_months"][0],
                            "last": backend.availability["monthly_months"][-1],
                            "complete_years": backend.monthly_full_years},
                "daily_black_marble": "2012 → quelques jours/semaines avant aujourd'hui (outil assess_event_impact)",
                "population_worldpop": "2015–2030 : WorldPop Global2 R2025A (2026–2030 = projections) ; 2000–2014 : WorldPop GP v1"},
            "definitions": {"sol": "Somme des radiances (nW/cm²/sr) des pixels de la zone",
                            "mean_rad": "SOL / nombre de pixels", "lit_km2": f"Surface des pixels > {LIT_THRESHOLD} nW/cm²/sr",
                            "lit_share": "Part de la surface éclairée (%)"},
            "caveats": ["NTL = proxy de l'activité et de l'électrification, pas une mesure directe du PIB ou de la pauvreté",
                        "Nuages, saisonnalité, torchères, halo urbain (blooming), bruit rural",
                        "Ne pas mélanger deux produits dans une même série (rupture de niveau)"],
            "notes": _base_notes()}


@tool("Découverte", core=True)
def list_admin_units(level: int = 1, iso3: Optional[str] = None) -> dict:
    """Liste les unités administratives (régions ADM1 ou districts ADM2) du pays avec leur superficie.
    Utile pour vérifier l'orthographe exacte d'une région avant une analyse ciblée.

    Args:
        level: Niveau administratif : 1 = régions/provinces, 2 = districts.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso, level = _iso(iso3), _level(level)
    df = backend.admin_units(iso, level).sort_values("area_km2", ascending=False)
    ds = S.add_dataset(df, "admin_units", f"Unités ADM{level} — {iso}", iso3=iso, level=level)
    return {"dataset_id": ds, "count": len(df), "units": _records(df, 60, ["unit", "area_km2"]), "notes": _base_notes()}

### 6.2 · Séries temporelles et anomalies

`detect_anomalies` illustre une bonne pratique : **ne jamais interpréter une baisse de luminosité sans regarder la couverture nuageuse**. La méthode est volontairement simple et explicable : désaisonnalisation par l'indice mensuel médian, tendance par médiane glissante, puis score z robuste (MAD).

In [ ]:
@tool("Séries temporelles", core=True)
def get_national_timeseries(start_year: Optional[int] = None, end_year: Optional[int] = None,
                            frequency: Literal["annual", "monthly"] = "annual",
                            source: Literal["auto", "vnl_annual", "monthly_mean"] = "auto",
                            iso3: Optional[str] = None) -> dict:
    """Série temporelle NATIONALE des lumières nocturnes : SOL, radiance moyenne, surface éclairée (et couverture
    nuageuse en mensuel). Calcule la variation totale, le taux de croissance annuel moyen (TCAM) et la tendance
    linéaire. Renvoie un dataset_id à passer à plot_timeseries.

    Args:
        start_year: Première année (défaut : 10 ans avant la dernière année disponible, 3 ans en mensuel).
        end_year: Dernière année (défaut : dernière année disponible).
        frequency: 'annual' ou 'monthly'.
        source: Produit pour l'annuel : 'auto', 'vnl_annual' (VNL masqué) ou 'monthly_mean' (moyenne des mensuels).
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso = _iso(iso3)
    start, end = _years(start_year, end_year, default_span=3 if frequency == "monthly" else 10)
    df = backend.national_series(iso, start, end, frequency, source)
    if df.empty:
        raise ValueError("Aucune donnée sur cette période.")
    ds = S.add_dataset(df, f"national_{frequency}", f"Série nationale {frequency} — {iso}", iso3=iso, frequency=frequency)
    first, last = df.iloc[0], df.iloc[-1]
    summary = {"period": f"{first.period} → {last.period}", "n_points": len(df),
               "sol_first": first.sol, "sol_last": last.sol,
               "sol_change_pct": 100 * (last.sol / first.sol - 1) if first.sol else None,
               "lit_km2_first": first.lit_km2, "lit_km2_last": last.lit_km2,
               "peak_period": df.loc[df.sol.idxmax(), "period"], "lowest_period": df.loc[df.sol.idxmin(), "period"]}
    notes = _base_notes() + [f"Source : {df['source'].iloc[0]}"]
    if frequency == "annual":
        n = int(last.period) - int(first.period)
        summary["cagr_pct"] = _cagr(first.sol, last.sol, n)
        tr = _trend(df.period.astype(int), df.sol)
        if tr:
            summary["trend_pct_per_year"] = tr["slope_pct_of_mean"]
            summary["trend_r2"], summary["trend_p_value"] = tr["r2"], tr["p_value"]
        S.add_kpi(f"SOL {last.period}", fmt(last.sol), "", summary["sol_change_pct"])
        if summary.get("cagr_pct") is not None:
            S.add_kpi("TCAM du SOL", f"{summary['cagr_pct']:.1f}", "%/an")
    else:
        low = df[df.coverage_pct < 70]
        summary["mean_coverage_pct"] = df.coverage_pct.mean()
        if len(low):
            notes.append(f"{len(low)} mois avec couverture sans nuage < 70 % : valeurs à interpréter avec prudence "
                         f"({', '.join(low.period.head(8))}).")
        S.add_kpi("Couverture moyenne", f"{df.coverage_pct.mean():.0f}", "%")
    S.add_kpi("Surface éclairée", fmt(last.lit_km2), "km²",
              100 * (last.lit_km2 / first.lit_km2 - 1) if first.lit_km2 else None)
    cols = ["period", "sol", "mean_rad", "lit_km2", "lit_share"] + (["coverage_pct"] if frequency == "monthly" else [])
    return {"dataset_id": ds, "summary": summary, "data": _records(df, 40 if frequency == "annual" else 14, cols),
            "notes": notes}


@tool("Séries temporelles")
def get_admin_timeseries(level: int = 1, start_year: Optional[int] = None, end_year: Optional[int] = None,
                         units: Optional[str] = None, iso3: Optional[str] = None) -> dict:
    """Séries annuelles PAR RÉGION (panel) : SOL, radiance moyenne et surface éclairée de chaque unité administrative,
    avec le TCAM de chacune. Permet de suivre une ou plusieurs régions nommées. Renvoie un dataset_id pour
    plot_timeseries (une courbe par région).

    Args:
        level: Niveau administratif : 1 = régions, 2 = districts.
        start_year: Première année (défaut : 10 ans avant la dernière disponible).
        end_year: Dernière année (défaut : dernière disponible).
        units: Noms de régions séparés par des virgules pour filtrer (ex. 'Kigali, Southern'). Vide = toutes.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso, level = _iso(iso3), _level(level)
    start, end = _years(start_year, end_year)
    df = backend.zonal_stats(iso, level, list(range(start, end + 1)))
    if units:
        keep = _match_units(units.split(","), df.unit.unique())
        if not keep:
            raise ValueError(f"Aucune région reconnue dans « {units} ». Utilisez list_admin_units pour les noms exacts.")
        df = df[df.unit.isin(keep)]
    df = df.sort_values(["unit", "year"])
    ds = S.add_dataset(df, "admin_timeseries", f"Séries ADM{level} {start}–{end} — {iso}", iso3=iso, level=level)
    g = df.groupby("unit").agg(sol_first=("sol", "first"), sol_last=("sol", "last")).reset_index()
    g["cagr_pct"] = [_cagr(a, b, end - start) for a, b in zip(g.sol_first, g.sol_last)]
    g = g.sort_values("cagr_pct", ascending=False)
    return {"dataset_id": ds, "period": f"{start}–{end}", "n_units": g.shape[0],
            "growth_by_unit": _records(g, 25), "notes": _base_notes() + [f"Source : {df['source'].iloc[0]}"]}


@tool("Séries temporelles", core=True)
def detect_anomalies(start_year: Optional[int] = None, end_year: Optional[int] = None,
                     z_threshold: float = 2.5, iso3: Optional[str] = None) -> dict:
    """Détecte les mois anormaux (chutes ou pics) de la luminosité nationale : désaisonnalisation, tendance par
    médiane glissante et score z robuste. Chaque anomalie est classée 'baisse probable (réelle)', 'hausse' ou
    'artefact probable (nuages)' selon la couverture sans nuage. Utile pour repérer coupures d'électricité, crises,
    conflits. Renvoie un dataset_id pour plot_timeseries.

    Args:
        start_year: Première année (défaut : 4 ans avant la dernière année mensuelle complète).
        end_year: Dernière année (défaut : dernière année mensuelle complète).
        z_threshold: Seuil du score z robuste (2 = sensible, 3 = strict).
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso = _iso(iso3)
    end = int(end_year) if end_year else max(backend.monthly_full_years)
    start = int(start_year) if start_year else end - 3
    df = backend.national_series(iso, start, end, "monthly").copy()
    s = df.sol.astype(float)
    df["month"] = df.period.str[5:7].astype(int)
    if len(df) >= 24:
        ratio = s / s.rolling(12, center=True, min_periods=6).median()
        seas = ratio.groupby(df.month).median()
        df["seasonal_index"] = df.month.map(seas / seas.mean())
    else:
        df["seasonal_index"] = 1.0
    deseason = s / df.seasonal_index
    baseline = deseason.rolling(7, center=True, min_periods=3).median()
    resid = np.log(deseason / baseline)
    mad = np.median(np.abs(resid - np.median(resid.dropna()))) or 1e-9
    df["z"] = (resid - np.median(resid.dropna())) / (1.4826 * mad)
    df["is_anomaly"] = df.z.abs() >= z_threshold
    def label(r):
        if not r.is_anomaly:
            return ""
        if r.coverage_pct < 70:
            return "artefact probable (nuages)"
        return "baisse probable (réelle)" if r.z < 0 else "hausse"
    df["anomaly_type"] = df.apply(label, axis=1)
    ds = S.add_dataset(df, "anomalies", f"Anomalies mensuelles — {iso}", iso3=iso, frequency="monthly")
    an = df[df.is_anomaly]
    S.add_kpi("Mois analysés", len(df))
    S.add_kpi("Anomalies détectées", len(an))
    return {"dataset_id": ds, "period": f"{df.period.iloc[0]} → {df.period.iloc[-1]}", "z_threshold": z_threshold,
            "anomalies": _records(an, 20, ["period", "sol", "z", "coverage_pct", "anomaly_type"]),
            "notes": _base_notes() + ["Une anomalie avec couverture < 70 % est plus probablement due aux nuages.",
                                      "Confirmer une baisse réelle par une source indépendante (presse, opérateur électrique)."]}

### 6.3 · Analyse spatiale infranationale

Au-delà du classement, `get_admin_stats` mesure la **concentration spatiale** de la lumière (indice de Gini entre unités, part de la première unité) et `compare_periods` teste une **convergence** : les régions les moins éclairées rattrapent-elles les autres ?

In [ ]:
@tool("Analyse spatiale", core=True)
def get_admin_stats(year: Optional[int] = None, level: int = 1,
                    sort_by: Literal["sol", "mean_rad", "lit_km2", "lit_share", "sol_share", "sol_density"] = "sol",
                    top_n: int = 10, ascending: bool = False, iso3: Optional[str] = None) -> dict:
    """Statistiques de lumières nocturnes PAR UNITÉ ADMINISTRATIVE pour une année : SOL, radiance moyenne, surface
    éclairée, part éclairée, part du SOL national, densité (SOL/km²), avec classement et concentration spatiale
    (Gini, part de la 1re unité). Renvoie un dataset_id pour plot_map (carte) ou plot_ranking (barres).

    Args:
        year: Année (défaut : dernière année disponible).
        level: Niveau administratif : 1 = régions, 2 = districts.
        sort_by: Indicateur de classement.
        top_n: Nombre d'unités renvoyées dans le résumé.
        ascending: True pour classer du moins au plus éclairé.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso, level = _iso(iso3), _level(level)
    year = int(year or backend.default_year())
    df = backend.zonal_stats(iso, level, [year])
    df["sol_share"] = 100 * df.sol / df.sol.sum()
    df["sol_density"] = df.sol / df.area_km2
    df = df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1)
    ds = S.add_dataset(df, "admin_stats", f"Statistiques ADM{level} {year} — {iso}", iso3=iso, level=level, year=year)
    by_sol = df.sort_values("sol", ascending=False)
    conc = {"gini_sol_between_units": _gini(df.sol), "top1_unit": by_sol.unit.iloc[0],
            "top1_share_pct": by_sol.sol_share.iloc[0], "top3_share_pct": by_sol.sol_share.head(3).sum(),
            "n_units": len(df), "national_lit_share_pct": 100 * df.lit_km2.sum() / df.area_km2.sum()}
    S.add_kpi("Unité la plus lumineuse", by_sol.unit.iloc[0], f"{by_sol.sol_share.iloc[0]:.0f} % du SOL")
    S.add_kpi("Gini spatial du SOL", f"{conc['gini_sol_between_units']:.2f}")
    return {"dataset_id": ds, "year": year, "level": level, "sorted_by": sort_by, "concentration": conc,
            "top": _records(df, top_n, ["rank", "unit", "sol", "mean_rad", "lit_km2", "lit_share", "sol_share", "sol_density"]),
            "notes": _base_notes() + [f"Source : {df['source'].iloc[0]}",
                                      "Gini proche de 1 = lumière concentrée dans peu d'unités."]}


@tool("Analyse spatiale", core=True)
def compare_periods(year_start: int, year_end: int, level: int = 1, iso3: Optional[str] = None) -> dict:
    """Compare deux années par unité administrative : variation absolue et en % du SOL, TCAM, variation de la surface
    éclairée ; identifie les plus fortes hausses et baisses et teste la convergence (les moins éclairées
    progressent-elles plus vite ?). Renvoie un dataset_id pour plot_map ou plot_ranking (metric='pct_change').

    Args:
        year_start: Année de départ (ex. 2015).
        year_end: Année d'arrivée (ex. 2023).
        level: Niveau administratif : 1 = régions, 2 = districts.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    from scipy import stats
    iso, level = _iso(iso3), _level(level)
    y0, y1 = sorted((int(year_start), int(year_end)))
    df = backend.zonal_stats(iso, level, [y0, y1])
    a = df[df.year == y0].set_index("unit_id")
    b = df[df.year == y1].set_index("unit_id")
    out = pd.DataFrame({"unit": a.unit, "sol_start": a.sol, "sol_end": b.sol,
                        "lit_km2_start": a.lit_km2, "lit_km2_end": b.lit_km2}).reset_index()
    out["abs_change"] = out.sol_end - out.sol_start
    out["pct_change"] = 100 * (out.sol_end / out.sol_start.replace(0, np.nan) - 1)
    out["cagr_pct"] = [_cagr(s, e, y1 - y0) for s, e in zip(out.sol_start, out.sol_end)]
    out["lit_km2_change"] = out.lit_km2_end - out.lit_km2_start
    out = out.sort_values("pct_change", ascending=False).reset_index(drop=True)
    ds = S.add_dataset(out, "change", f"Variation {y0}→{y1} ADM{level} — {iso}", iso3=iso, level=level,
                       year_start=y0, year_end=y1)
    nat = 100 * (out.sol_end.sum() / out.sol_start.sum() - 1)
    valid = out.dropna(subset=["cagr_pct"])
    conv = None
    if len(valid) >= 5:
        rho, p = stats.spearmanr(np.log(valid.sol_start + 1e-9), valid.cagr_pct)
        conv = {"spearman_initial_level_vs_growth": rho, "p_value": p,
                "reading": "convergence (les moins éclairées rattrapent)" if rho < -0.3 and p < 0.1 else
                           "divergence (les plus éclairées creusent l'écart)" if rho > 0.3 and p < 0.1 else
                           "pas de relation nette"}
    S.add_kpi(f"Variation nationale {y0}→{y1}", f"{nat:+.1f}", "%")
    S.add_kpi("Plus forte hausse", out.unit.iloc[0], f"{out['pct_change'].iloc[0]:+.0f} %")
    return {"dataset_id": ds, "period": f"{y0}→{y1}", "national_sol_change_pct": nat,
            "top_gainers": _records(out, 5, ["unit", "sol_start", "sol_end", "pct_change", "cagr_pct"]),
            "top_decliners": _records(out.iloc[::-1], 5, ["unit", "sol_start", "sol_end", "pct_change", "cagr_pct"]),
            "convergence_test": conv,
            "notes": _base_notes() + ["Les petites unités peu éclairées ont des variations en % très volatiles."]}

### 6.4 · Population, développement et validation

La journée 4 de l'atelier se termine par une question décisive : **le proxy NTL est-il utilisable pour la diffusion dans votre pays, ou reste-t-il un outil de diagnostic ?** Ces outils y répondent avec des éléments chiffrés : population vivant dans l'obscurité (WorldPop), corrélation avec des indicateurs officiels de la Banque mondiale et comparaison entre pays pairs.

In [ ]:
WB_ALIASES = {
    "gdp": ("NY.GDP.MKTP.KD", "PIB (USD constants 2015)"),
    "gdp_per_capita": ("NY.GDP.PCAP.KD", "PIB par habitant (USD constants 2015)"),
    "electricity_access": ("EG.ELC.ACCS.ZS", "Accès à l'électricité (% de la population)"),
    "electricity_access_rural": ("EG.ELC.ACCS.RU.ZS", "Accès à l'électricité, rural (%)"),
    "population": ("SP.POP.TOTL", "Population totale"),
    "urban_share": ("SP.URB.TOTL.IN.ZS", "Population urbaine (%)"),
    "electricity_consumption": ("EG.USE.ELEC.KH.PC", "Consommation d'électricité (kWh/hab.)"),
}


def _worldbank(iso, code, start, end):
    def compute():
        url = f"https://api.worldbank.org/v2/country/{iso}/indicator/{code}?format=json&per_page=200&date={start}:{end}"
        js = requests.get(url, timeout=30).json()
        if len(js) < 2 or not js[1]:
            raise ValueError(f"Aucune donnée Banque mondiale pour {code} / {iso}.")
        rows = [(int(r["date"]), r["value"]) for r in js[1] if r["value"] is not None]
        return pd.DataFrame(rows, columns=["year", "value"]).sort_values("year").reset_index(drop=True)
    return CACHE.get(["wb", iso, code, start, end, dt.date.today().isoformat()[:7]], compute).copy()


@tool("Population & validation")
def get_population_light_stats(year: Optional[int] = None, level: int = 1, iso3: Optional[str] = None) -> dict:
    """Croise lumières nocturnes et population (WorldPop Global2, 100 m) par unité administrative : population, population
    vivant dans des pixels NON éclairés (proxy du déficit d'accès à l'électricité), et SOL pour 1 000 habitants.
    Renvoie un dataset_id pour plot_map / plot_ranking (metric='pop_unlit_share' ou 'sol_per_1000').

    Args:
        year: Année NTL (défaut : dernière disponible). Population WorldPop Global2 de la même année (2015–2030).
        level: Niveau administratif : 1 = régions, 2 = districts.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso, level = _iso(iso3), _level(level)
    year = int(year or backend.default_year())
    pop = backend.population_stats(iso, level, year)
    z = backend.zonal_stats(iso, level, [year])[["unit_id", "sol", "lit_share"]]
    df = pop.merge(z, on="unit_id", how="left")
    df["pop_unlit_share"] = 100 * df.pop_unlit / df["pop"].replace(0, np.nan)
    df["sol_per_1000"] = 1000 * df.sol / df["pop"].replace(0, np.nan)
    df = df.sort_values("pop_unlit_share", ascending=False).reset_index(drop=True)
    ds = S.add_dataset(df, "population_light", f"Population et lumière ADM{level} {year} — {iso}",
                       iso3=iso, level=level, year=year)
    nat_share = 100 * df.pop_unlit.sum() / df["pop"].sum()
    S.add_kpi("Population en zone non éclairée", f"{nat_share:.0f}", "%")
    S.add_kpi("Population (WorldPop)", fmt(df["pop"].sum()), f"en {int(df.pop_year.iloc[0])}")
    notes = _base_notes() + [
        "Un pixel non éclairé ne signifie pas absence d'électricité (petites consommations, solaire hors réseau, "
        "éclairage éteint la nuit) : c'est un proxy, à comparer au taux d'accès officiel (get_worldbank_indicator)."]
    src_pop, py = df.pop_source.iloc[0], int(df.pop_year.iloc[0])
    notes.append(f"Population : {src_pop}, année {py}.")
    if py != year:
        notes.append(f"Population de {py} appliquée aux lumières de {year} (année la plus proche disponible).")
    if "projection" in src_pop:
        notes.append("Population projetée (au-delà de 2025) : ce n'est pas une estimation observée.")
    return {"dataset_id": ds, "year": year, "population_year": py, "population_source": src_pop,
            "national": {"population": df["pop"].sum(), "pop_unlit": df.pop_unlit.sum(), "pop_unlit_share_pct": nat_share},
            "by_unit": _records(df, 15, ["unit", "pop", "pop_unlit_share", "sol_per_1000", "lit_share"]), "notes": notes}


@tool("Population & validation")
def get_worldbank_indicator(indicator: str = "gdp", start_year: Optional[int] = None,
                            end_year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """Récupère un indicateur officiel de la Banque mondiale (API WDI). Alias acceptés : gdp, gdp_per_capita,
    electricity_access, electricity_access_rural, population, urban_share, electricity_consumption — ou un code
    WDI brut (ex. 'EG.ELC.ACCS.ZS').

    Args:
        indicator: Alias ou code WDI de l'indicateur.
        start_year: Première année (défaut : 2012).
        end_year: Dernière année (défaut : année en cours).
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso = _iso(iso3)
    code, label = WB_ALIASES.get(indicator, (indicator, indicator))
    start, end = int(start_year or 2012), int(end_year or dt.date.today().year)
    df = _worldbank(iso, code, start, end)
    ds = S.add_dataset(df.rename(columns={"value": "indicator"}).assign(period=lambda d: d.year.astype(str)),
                       "worldbank", f"{label} — {iso}", iso3=iso, indicator=label)
    return {"dataset_id": ds, "indicator": label, "code": code, "data": _records(df, 40),
            "source": "Banque mondiale, World Development Indicators (API v2)"}


@tool("Population & validation")
def correlate_ntl_with_indicator(indicator: str = "gdp", start_year: Optional[int] = None,
                                 end_year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """VALIDATION du proxy : corrèle la série annuelle nationale du SOL avec un indicateur officiel de la Banque
    mondiale (alias : gdp, gdp_per_capita, electricity_access, population, urban_share…). Calcule r de Pearson,
    rho de Spearman et l'élasticité log-log (variation en % de l'indicateur pour +1 % de SOL). Renvoie un dataset_id
    pour plot_scatter (x='sol', y='indicator').

    Args:
        indicator: Alias ou code WDI de l'indicateur.
        start_year: Première année (défaut : première année NTL disponible).
        end_year: Dernière année (défaut : dernière année NTL disponible).
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    from scipy import stats
    iso = _iso(iso3)
    start, end = _years(start_year, end_year, default_span=15)
    code, label = WB_ALIASES.get(indicator, (indicator, indicator))
    ntl = backend.national_series(iso, start, end, "annual")[["period", "sol"]]
    ntl["year"] = ntl.period.astype(int)
    wb = _worldbank(iso, code, start, end).rename(columns={"value": "indicator"})
    df = ntl.merge(wb, on="year").dropna()
    if len(df) < 4:
        raise ValueError(f"Seulement {len(df)} années communes : corrélation impossible. Élargissez la période.")
    pr = stats.pearsonr(df.sol, df.indicator)
    sr = stats.spearmanr(df.sol, df.indicator)
    el = stats.linregress(np.log(df.sol), np.log(df.indicator)) if (df.indicator > 0).all() else None
    ds = S.add_dataset(df, "correlation", f"SOL vs {label} — {iso}", iso3=iso, indicator=label)
    S.add_kpi("Corrélation r (Pearson)", f"{pr[0]:.2f}")
    if el:
        S.add_kpi("Élasticité log-log", f"{el.slope:.2f}")
    return {"dataset_id": ds, "indicator": label, "n_years": len(df), "years": f"{df.year.min()}–{df.year.max()}",
            "pearson_r": pr[0], "pearson_p": pr[1], "spearman_rho": sr[0],
            "elasticity_loglog": el.slope if el else None, "elasticity_r2": el.rvalue ** 2 if el else None,
            "notes": _base_notes() + [
                "Deux séries en tendance sont souvent corrélées sans lien causal (corrélation fallacieuse).",
                "Avec moins de 10 points, l'incertitude est très forte : conclure avec prudence.",
                "La validation la plus convaincante est infranationale (NTL vs statistiques régionales officielles)."]}


@tool("Population & validation")
def compare_countries(iso3_list: str, year: Optional[int] = None) -> dict:
    """Compare plusieurs pays pour une année : SOL, surface éclairée, radiance moyenne, SOL/km², SOL pour 1 000
    habitants et accès à l'électricité (Banque mondiale). Idéal pour se situer par rapport à des pays pairs.
    Renvoie un dataset_id pour plot_ranking.

    Args:
        iso3_list: Codes ISO3 séparés par des virgules, ex. 'RWA,BDI,UGA,KEN'.
        year: Année (défaut : dernière année disponible).
    """
    year = int(year or backend.default_year())
    rows = []
    for code in [c.strip().upper() for c in iso3_list.split(",") if c.strip()][:8]:
        iso = _iso(code)
        nat = backend.national_series(iso, year, year, "annual").iloc[0]
        area = float(backend.admin_units(iso, 0).area_km2.sum())
        r = dict(iso3=iso, unit=country_name(iso), sol=nat.sol, mean_rad=nat.mean_rad, lit_km2=nat.lit_km2,
                 lit_share=nat.lit_share, sol_density=nat.sol / area)
        for key, col in (("population", "population"), ("electricity_access", "electricity_access")):
            try:
                w = _worldbank(iso, WB_ALIASES[key][0], year - 3, year)
                r[col] = w.value.iloc[-1]
            except Exception:
                r[col] = None
        if r["population"]:
            r["sol_per_capita_1000"] = 1000 * r["sol"] / r["population"]
        rows.append(r)
    df = pd.DataFrame(rows).sort_values("sol", ascending=False).reset_index(drop=True)
    ds = S.add_dataset(df, "countries", f"Comparaison de pays {year}", year=year)
    return {"dataset_id": ds, "year": year, "comparison": _records(df, 10),
            "notes": _base_notes() + ["Comparer de préférence des indicateurs relatifs (par km², par habitant)."]}

### 6.5 · Événements et contrôle qualité

In [ ]:
@tool("Événements & qualité")
def assess_event_impact(event_date: str, days_before: int = 30, days_after: int = 30, level: int = 1,
                        iso3: Optional[str] = None) -> dict:
    """Mesure l'impact d'un événement daté (coupure d'électricité, inondation, conflit, cyclone, fête…) avec les
    images JOURNALIÈRES NASA Black Marble : médiane des nuits de bonne qualité avant vs après, par unité
    administrative. Renvoie un dataset_id pour plot_map / plot_ranking (metric='pct_change').

    Args:
        event_date: Date de l'événement au format AAAA-MM-JJ.
        days_before: Nombre de jours de la fenêtre avant l'événement.
        days_after: Nombre de jours de la fenêtre après l'événement.
        level: Niveau administratif : 1 = régions, 2 = districts.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso, level = _iso(iso3), _level(level)
    pd.Timestamp(event_date)                       # valide le format
    df = backend.event_impact(iso, level, event_date, int(days_before), int(days_after))
    df["pct_change"] = 100 * (df.sol_after / df.sol_before.replace(0, np.nan) - 1)
    df = df.sort_values("pct_change").reset_index(drop=True)
    ds = S.add_dataset(df, "event", f"Impact de l'événement du {event_date} — {iso}", iso3=iso, level=level)
    nat = 100 * (df.sol_after.sum() / df.sol_before.sum() - 1)
    S.add_kpi("Variation nationale après l'événement", f"{nat:+.1f}", "%")
    S.add_kpi("Unité la plus touchée", df.unit.iloc[0], f"{df['pct_change'].iloc[0]:+.0f} %")
    return {"dataset_id": ds, "event_date": event_date, "window_days": [days_before, days_after],
            "national_change_pct": nat, "images_before": int(df.n_images_before.iloc[0]),
            "images_after": int(df.n_images_after.iloc[0]),
            "most_affected": _records(df, 8, ["unit", "sol_before", "sol_after", "pct_change", "coverage_after_pct"]),
            "notes": _base_notes() + ["Variations de ±10 % fréquentes sans événement (lune, nuages, angle de visée).",
                                      "Comparer avec la même fenêtre l'année précédente pour isoler l'effet."]}


GAS_FLARING = {"NGA", "DZA", "LBY", "EGY", "AGO", "GAB", "COG", "GNQ", "CMR", "TCD", "SDN", "SSD", "TUN", "CIV", "GHA", "COD"}


@tool("Événements & qualité", core=False)
def get_quality_report(year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """Rapport de QUALITÉ des données NTL pour une année : couverture sans nuage mois par mois, mois douteux,
    distribution des radiances (médiane, p90, p99, maximum) et artefacts probables (torchères, saturation, nuages).
    À utiliser avant d'interpréter une baisse ou une valeur surprenante.

    Args:
        year: Année (défaut : dernière année disponible).
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso = _iso(iso3)
    year = int(year or backend.default_year())
    q = backend.quality_report(iso, year)
    cov = q["monthly_coverage"]
    ds = S.add_dataset(cov, "quality", f"Couverture mensuelle {year} — {iso}", iso3=iso, frequency="monthly")
    p = q["percentiles"]
    flags = []
    bad = cov[cov.coverage_pct < 70]
    if len(bad):
        flags.append(f"Nuages : {len(bad)} mois avec couverture < 70 % ({', '.join(bad.period)}).")
    if p.get("max") and p.get("p99") and p["max"] > 5 * p["p99"]:
        flags.append("Valeurs extrêmes isolées (max ≫ p99) : torchères, sites industriels ou pixels saturés possibles.")
    if iso in GAS_FLARING:
        flags.append("Pays avec torchage de gaz connu : vérifier les pixels très brillants hors des villes.")
    S.add_kpi("Couverture moyenne", f"{cov.coverage_pct.mean():.0f}", "%")
    return {"dataset_id": ds, "year": year, "radiance_percentiles_lit_pixels": p,
            "monthly_coverage": _records(cov, 12), "flags": flags or ["Aucun artefact majeur détecté automatiquement."],
            "notes": _base_notes() + ["Le blooming (halo urbain) et le bruit rural ne sont pas détectables par ce test simple."]}

### 6.6 · Visualisation et export

Les outils graphiques ne reçoivent **pas de données** du LLM : seulement un `dataset_id`. Le LLM ne peut donc pas « inventer » les valeurs tracées. Les graphiques sont interactifs (survol, zoom), enregistrés en HTML dans `ntl_agent_outputs/figures/` et repris dans le rapport final.

In [ ]:
PCT_METRICS = {"pct_change", "cagr_pct", "lit_share", "sol_share", "pop_unlit_share", "coverage_pct",
               "electricity_access", "abs_change", "lit_km2_change"}
DIVERGING = {"pct_change", "cagr_pct", "abs_change", "lit_km2_change", "z"}
WARM = ["#FBEFD9", "#F5C242", "#D49A00", "#C4621D", "#B83B2E"]   # « plus = moins favorable »
NEGATIVE_METRICS = {"pop_unlit_share", "pop_unlit"}


def _style(fig, title, ytitle=None, xtitle=None, height=470):
    src = f"Source : {backend.name}"
    if not backend.synthetic:
        src = "Sources : NOAA/EOG VIIRS, NASA Black Marble, WorldPop, geoBoundaries, Banque mondiale — via Earth Engine"
    src += f" · seuil éclairé {LIT_THRESHOLD} nW/cm²/sr · Agent NTL STG17"
    fig.update_layout(title=dict(text=f"<b>{title}</b>"), height=height,
                      legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1))
    if ytitle is not None:
        fig.update_yaxes(title_text=ytitle)
    if xtitle is not None:
        fig.update_xaxes(title_text=xtitle)
    fig.add_annotation(text=f"<i>{src}</i>", xref="paper", yref="paper", x=0, y=-0.13, xanchor="left",
                       yanchor="top", showarrow=False, font=dict(size=10, color=C["slate"]))
    if backend.synthetic:
        fig.add_annotation(text="DÉMO — DONNÉES SYNTHÉTIQUES", xref="paper", yref="paper", x=0.5, y=0.5,
                           showarrow=False, textangle=-18, font=dict(size=30, color="rgba(184,59,46,0.16)"))
    return fig


def _require(df, metric):
    if metric not in df.columns:
        num = [c for c in df.select_dtypes("number").columns if c not in ("n_px", "year", "month", "rank")]
        raise ValueError(f"Colonne « {metric} » absente de ce dataset. Colonnes numériques possibles : {num}")


def _polygons(g):
    """Extrait uniquement les polygones d'une géométrie. Après simplification, Earth Engine peut renvoyer des
    GeometryCollection mêlant polygones et segments (LineString) : ces segments n'ont pas de surface à colorier."""
    from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
    if g is None or g.is_empty:
        return []
    if isinstance(g, Polygon):
        return [g]
    if isinstance(g, (MultiPolygon, GeometryCollection)):
        return [p for part in g.geoms for p in _polygons(part)]
    return []                                   # LineString, Point… ignorés


def _one_year(df, year):
    if "year" in df.columns and df.year.nunique() > 1:
        return df[df.year == (int(year) if year else df.year.max())]
    return df


@tool("Visualisation", core=True)
def plot_timeseries(dataset_id: str, metric: str = "sol", title: Optional[str] = None, top_n: int = 8) -> dict:
    """Trace l'évolution d'un indicateur dans le temps à partir d'un dataset_id (série nationale annuelle ou
    mensuelle, anomalies, panel de régions). Ajoute la tendance, les anomalies et la couverture nuageuse quand
    elles existent.

    Args:
        dataset_id: Identifiant renvoyé par un outil de données (ex. 'ds02_national_annual').
        metric: Colonne à tracer : sol, mean_rad, lit_km2, lit_share, coverage_pct…
        title: Titre du graphique (court, informatif).
        top_n: Pour un panel de régions : nombre de régions tracées (les plus élevées la dernière année).
    """
    ds = S.get(dataset_id)
    df = ds["df"].copy()
    _require(df, metric)
    title = title or ds["title"]
    fig = go.Figure()
    if "unit" in df.columns and "year" in df.columns and df.unit.nunique() > 1:
        last = df[df.year == df.year.max()].nlargest(top_n, metric).unit.tolist()
        for i, u in enumerate(last):
            sub = df[df.unit == u].sort_values("year")
            fig.add_scatter(x=sub.year, y=sub[metric], mode="lines+markers", name=u,
                            line=dict(width=3 if i == 0 else 2, color=SERIES[i % len(SERIES)]), marker=dict(size=6))
        xt = "Année"
    else:
        xcol = "period" if "period" in df.columns else "year"
        d = df.sort_values(xcol)
        monthly = d[xcol].astype(str).str.len().eq(7).all()
        xs = pd.to_datetime(d[xcol].astype(str)) if monthly else d[xcol].astype(int)
        if not monthly:
            fig.update_xaxes(dtick=1, tickformat="d")
        if monthly and "coverage_pct" in d.columns and metric != "coverage_pct":
            fig.add_bar(x=xs, y=d.coverage_pct, name="Couverture sans nuage (%)", yaxis="y2",
                        marker_color="rgba(169,181,176,0.35)", hovertemplate="%{y:.0f} %<extra>couverture</extra>")
            fig.update_layout(yaxis2=dict(overlaying="y", side="right", range=[0, 300], showgrid=False,
                                          showticklabels=False, title=None))
        fig.add_scatter(x=xs, y=d[metric], mode="lines+markers", name=LABELS.get(metric, metric),
                        line=dict(color=C["green"], width=3), marker=dict(size=5 if monthly else 8, color=C["deep"]),
                        fill="tozeroy", fillcolor="rgba(0,168,106,0.07)")
        if not monthly and len(d) >= 4:
            xn = np.arange(len(d))
            coef = np.polyfit(xn, d[metric].astype(float), 1)
            fig.add_scatter(x=xs, y=np.polyval(coef, xn), mode="lines", name="Tendance linéaire",
                            line=dict(color=C["ochre"], width=2, dash="dash"))
        if "is_anomaly" in d.columns:
            colors = {"baisse probable (réelle)": C["brick"], "artefact probable (nuages)": C["ochre"], "hausse": C["teal"]}
            for lab, col in colors.items():
                a = d[d.anomaly_type == lab]
                if len(a):
                    fig.add_scatter(x=pd.to_datetime(a.period), y=a[metric], mode="markers", name=lab,
                                    marker=dict(size=15, color=col, symbol="circle-open", line=dict(width=3)))
        fig.add_annotation(x=xs.iloc[-1], y=d[metric].iloc[-1], text=f"<b>{fmt(d[metric].iloc[-1])}</b>",
                           showarrow=False, yshift=16, font=dict(color=C["deep"]))
        xt = "Mois" if monthly else "Année"
    _style(fig, title, LABELS.get(metric, metric), xt)
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Graphique créé ; il s'affichera sous la réponse."}


@tool("Visualisation", core=True)
def plot_ranking(dataset_id: str, metric: str = "sol", top_n: int = 15, title: Optional[str] = None,
                 year: Optional[int] = None) -> dict:
    """Diagramme en barres horizontales classant les unités (régions, districts ou pays) selon un indicateur.
    Couleurs divergentes (vert/rouge) pour les variations.

    Args:
        dataset_id: Identifiant d'un dataset par unité (get_admin_stats, compare_periods, get_population_light_stats…).
        metric: Colonne à classer : sol, mean_rad, lit_share, sol_share, pct_change, pop_unlit_share…
        top_n: Nombre de barres.
        title: Titre du graphique.
        year: Pour un panel multi-années : année à représenter (défaut : la plus récente).
    """
    ds = S.get(dataset_id)
    df = _one_year(ds["df"].copy(), year)
    _require(df, metric)
    title = title or f"{LABELS.get(metric, metric)} — {ds['title']}"
    d = df.dropna(subset=[metric]).sort_values(metric, ascending=False)
    div = metric in DIVERGING or (d[metric] < 0).any()
    if len(d) > top_n:
        d = pd.concat([d.head(top_n - top_n // 3), d.tail(top_n // 3)]) if div else d.head(top_n)
    d = d.iloc[::-1]
    if div:
        colors = [C["green"] if v >= 0 else C["brick"] for v in d[metric]]
    else:
        ramp = WARM if metric in NEGATIVE_METRICS else RAMP
        idx = np.linspace(1, len(ramp) - 1, len(d)).round().astype(int)
        colors = [ramp[i] for i in idx]
    suffix = " %" if metric in PCT_METRICS and metric not in ("abs_change", "lit_km2_change") else ""
    fig = go.Figure(go.Bar(x=d[metric], y=d.unit, orientation="h", marker_color=colors,
                           text=[f"{fmt(v)}{suffix}" for v in d[metric]], textposition="outside", cliponaxis=False,
                           hovertemplate="%{y}<br>%{x:,.2f}<extra></extra>"))
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(gridcolor="rgba(0,0,0,0)")
    _style(fig, title, None, LABELS.get(metric, metric), height=max(380, 30 * len(d) + 160))
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Graphique créé ; il s'affichera sous la réponse."}


@tool("Visualisation", core=True)
def plot_map(dataset_id: str, metric: str = "sol", title: Optional[str] = None, year: Optional[int] = None) -> dict:
    """Carte choroplèthe des unités administratives colorées selon un indicateur (échelle logarithmique automatique
    pour les indicateurs très asymétriques comme le SOL ; échelle divergente pour les variations).

    Args:
        dataset_id: Identifiant d'un dataset par unité administrative (get_admin_stats, compare_periods,
            get_population_light_stats, assess_event_impact, get_admin_timeseries).
        metric: Colonne à cartographier : sol, mean_rad, lit_share, sol_density, pct_change, pop_unlit_share…
        title: Titre de la carte.
        year: Pour un panel multi-années : année à cartographier (défaut : la plus récente).
    """
    ds = S.get(dataset_id)
    meta = ds["meta"]
    if "level" not in meta or "unit_id" not in ds["df"].columns:
        raise ValueError("Ce dataset n'est pas découpé par unité administrative : utilisez plot_ranking ou plot_timeseries.")
    df = _one_year(ds["df"].copy(), year)
    _require(df, metric)
    gj = backend.geojson(meta["iso3"], meta["level"])
    title = title or f"{LABELS.get(metric, metric)} — {ds['title']}"
    v = df[metric].astype(float)
    div = metric in DIVERGING or (v < 0).any()
    cb = dict(title=dict(text=LABELS.get(metric, metric).replace(" (", "<br>("), side="top"), thickness=14, len=0.75)
    if div:
        lim = float(np.nanmax(np.abs(v))) or 1
        df["_c"] = v
        kw = dict(color_continuous_scale=[[0, C["brick"]], [0.5, "#F4F1EA"], [1, C["green"]]], range_color=(-lim, lim))
    elif v.min() > 0 and v.max() / v.min() > 50:
        df["_c"] = np.log10(v)
        ticks = np.arange(math.floor(df._c.min()), math.ceil(df._c.max()) + 1)
        cb.update(tickvals=ticks, ticktext=[fmt(10.0 ** float(t)) for t in ticks])
        kw = dict(color_continuous_scale=RAMP)
    else:
        df["_c"] = v
        kw = dict(color_continuous_scale=WARM if metric in NEGATIVE_METRICS else RAMP)
    # Carte dessinée en polygones Plotly « cartésiens » : aucun fond de carte à télécharger,
    # donc fonctionne aussi hors ligne (Kaggle sans Internet, réseau d'entreprise filtré…).
    from plotly.colors import sample_colorscale
    scale = kw["color_continuous_scale"]
    lo, hi = kw.get("range_color", (float(np.nanmin(df._c)), float(np.nanmax(df._c))))
    span = (hi - lo) or 1.0
    vals = dict(zip(df.unit_id, zip(df._c, v, df.unit)))
    fig = go.Figure()
    lats = []
    for f in gj["features"]:
        uid = f["properties"]["unit_id"]
        g = shape(f["geometry"])
        polys = _polygons(g)
        c, raw, name = vals.get(uid, (np.nan, np.nan, f["properties"].get("unit", uid)))
        col = "#E6EBE8" if pd.isna(c) else sample_colorscale(scale, [min(max((c - lo) / span, 0), 1)])[0]
        tip = f"<b>{name}</b><br>{LABELS.get(metric, metric)} : {fmt(raw)}"
        for i, poly in enumerate(polys):
            xs, ys = poly.exterior.xy
            lats += list(ys)
            fig.add_scatter(x=list(xs), y=list(ys), fill="toself", fillcolor=col, mode="lines",
                            line=dict(color="white", width=0.8), hoveron="fills", text=tip, hoverinfo="text",
                            showlegend=False, legendgroup=uid)
    fig.add_scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                    marker=dict(colorscale=scale, cmin=lo, cmax=hi, color=[lo], showscale=True, colorbar=cb))
    if len(df) <= 35:                               # étiquettes au centre des unités
        pts = []
        for f in gj["features"]:
            ps = _polygons(shape(f["geometry"]))
            if ps:
                pts.append((max(ps, key=lambda q: q.area).representative_point(), f["properties"].get("unit", "")))
        fig.add_scatter(x=[p.x for p, _ in pts], y=[p.y for p, _ in pts], text=[n for _, n in pts], mode="text",
                        textfont=dict(size=10, color=C["ink"]), hoverinfo="skip", showlegend=False)
    lat_mid = float(np.mean(lats)) if lats else 0.0
    fig.update_xaxes(visible=False, showgrid=False)
    fig.update_yaxes(visible=False, showgrid=False, scaleanchor="x", scaleratio=1 / max(math.cos(math.radians(lat_mid)), 0.2))
    fig.update_layout(margin=dict(l=10, r=10, t=70, b=70), hovermode="closest")
    _style(fig, title, height=580)
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Carte créée ; elle s'affichera sous la réponse."}


@tool("Visualisation")
def plot_scatter(dataset_id: str, x: str = "sol", y: str = "indicator", log_scale: bool = True,
                 title: Optional[str] = None) -> dict:
    """Nuage de points entre deux colonnes d'un dataset (ex. SOL vs PIB après correlate_ntl_with_indicator), avec
    droite d'ajustement et étiquettes (années ou unités).

    Args:
        dataset_id: Identifiant du dataset.
        x: Colonne en abscisse.
        y: Colonne en ordonnée.
        log_scale: Axes logarithmiques (recommandé pour SOL et PIB).
        title: Titre du graphique.
    """
    ds = S.get(dataset_id)
    df = _one_year(ds["df"].copy(), None).dropna(subset=[x, y])
    _require(df, x); _require(df, y)
    lab = df["year"].astype(str) if "year" in df.columns and df.year.nunique() > 1 else df.get("unit", df.index.astype(str))
    title = title or f"{LABELS.get(y, ds['meta'].get('indicator', y))} vs {LABELS.get(x, x)}"
    fig = go.Figure(go.Scatter(x=df[x], y=df[y], mode="markers+text", text=lab, textposition="top center",
                               marker=dict(size=11, color=C["green"], line=dict(color="white", width=1)),
                               textfont=dict(size=10, color=C["slate"]), name="observations"))
    xx, yy = df[x].astype(float), df[y].astype(float)
    ok = (xx > 0) & (yy > 0) if log_scale else pd.Series(True, index=df.index)
    if ok.sum() >= 3:
        fx, fy = (np.log(xx[ok]), np.log(yy[ok])) if log_scale else (xx[ok], yy[ok])
        coef = np.polyfit(fx, fy, 1)
        grid = np.linspace(fx.min(), fx.max(), 50)
        fig.add_scatter(x=np.exp(grid) if log_scale else grid,
                        y=np.exp(np.polyval(coef, grid)) if log_scale else np.polyval(coef, grid), mode="lines",
                        line=dict(color=C["ochre"], dash="dash"), name=f"ajustement (pente {coef[0]:.2f})")
    if log_scale:
        fig.update_xaxes(type="log")
        fig.update_yaxes(type="log")
    _style(fig, title, LABELS.get(y, ds["meta"].get("indicator", y)), LABELS.get(x, x))
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Graphique créé ; il s'affichera sous la réponse."}


@tool("Visualisation")
def show_night_image(year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """Affiche l'image satellite des lumières nocturnes du pays pour une année (échelle logarithmique, fond sombre).
    Idéal pour une première impression visuelle ou pour repérer torchères et halos urbains.

    Args:
        year: Année (défaut : dernière disponible).
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso = _iso(iso3)
    year = int(year or backend.default_year())
    png = backend.thumbnail(iso, year)
    title = f"{country_name(iso)} de nuit — {year}"
    fig_id = S.add_figure(png, title, kind="image")
    return {"figure_id": fig_id, "title": title, "message": "Image créée ; elle s'affichera sous la réponse."}


@tool("Export")
def export_dataset(dataset_id: str, file_format: Literal["csv", "xlsx"] = "csv") -> dict:
    """Exporte un dataset de la session en CSV ou Excel dans le dossier de sorties (pour publication ou réutilisation).

    Args:
        dataset_id: Identifiant du dataset à exporter.
        file_format: 'csv' ou 'xlsx'.
    """
    ds = S.get(dataset_id)
    path = OUT / "data" / f"{dataset_id}.{file_format}"
    df = ds["df"].drop(columns=[c for c in ("n_px",) if c in ds["df"].columns])
    (df.to_csv(path, index=False) if file_format == "csv" else df.to_excel(path, index=False))
    return {"path": str(path), "rows": len(df), "columns": list(df.columns)}


@tool("Export")
def list_session_datasets() -> dict:
    """Liste les datasets déjà calculés dans la session (identifiant, titre, colonnes), pour les réutiliser dans un
    graphique ou un export sans recalculer."""
    return {"datasets": [{"dataset_id": k, "title": v["title"], "rows": len(v["df"]),
                          "columns": [c for c in v["df"].columns if c != "n_px"]} for k, v in S.datasets.items()][-15:]}

In [ ]:
def show_toolbox():
    fam_color = {"Découverte": C["green"], "Séries temporelles": C["deep"], "Analyse spatiale": C["teal"],
                 "Population & validation": C["ochre"], "Événements & qualité": C["terra"],
                 "Visualisation": C["brick"], "Export": C["slate"], "Extension": C["forest"]}
    html = ""
    for fam in dict.fromkeys(t["family"] for t in TOOLS.values()):
        cards = ""
        for name, t in TOOLS.items():
            if t["family"] != fam:
                continue
            desc = t["schema"]["function"]["description"]
            short = desc.split(". ")[0][:150]
            core = ' <span class="ntl-pill" style="font-size:10px;padding:0 6px">core</span>' if t["core"] else ""
            cards += f'<div class="ntl-tool"><b>{name}</b>{core}<div class="ntl-muted">{short}</div></div>'
        col = fam_color.get(fam, C["green"])
        html += (f'<div style="margin:12px 0 6px;font-weight:700;color:{col};letter-spacing:2px;'
                 f'text-transform:uppercase;font-size:12px">● {fam}</div><div class="ntl-grid">{cards}</div>')
    n_core = sum(t["core"] for t in TOOLS.values())
    ui.card(f"{len(TOOLS)} outils disponibles ({n_core} dans le jeu « core »)", html, kicker="06 · Boîte à outils")

show_toolbox()

## 07 · Tester les outils à la main (sans LLM)

Avant de confier les outils à un agent, **vérifiez-les vous-même**. C'est aussi ce que « voit » le LLM : un dictionnaire JSON compact. Si un humain ne comprend pas le résultat d'un outil, le LLM non plus.

In [ ]:
S.new_turn()
profile = get_country_profile()
series = get_national_timeseries()
ui.card(f"{profile['country']} — premiers résultats", ui.kpis(S.kpis) +
        f'<div class="ntl-muted">Dataset créé : <code>{series["dataset_id"]}</code> · {series["notes"][-1]}</div>',
        kicker="07 · Test manuel")
fig_id = plot_timeseries(series["dataset_id"], "sol", f"Somme des radiances — {profile['country']}")["figure_id"]
S.figures[fig_id]["obj"].show()

In [ ]:
# Indépendante de l'ordre d'exécution : recalcule ce qui manque.
if "profile" not in globals():
    S.new_turn()
    profile = get_country_profile()
stats = get_admin_stats()
S.figures[plot_map(stats["dataset_id"], "mean_rad")["figure_id"]]["obj"].show()
img = S.figures[show_night_image()["figure_id"]]["obj"]
ui.card(f"{profile['country']} de nuit",
        f'<div style="background:#000814;border-radius:10px;padding:12px;text-align:center">'
        f'<img src="data:image/png;base64,{base64.b64encode(img).decode()}" style="max-width:100%;max-height:520px"></div>',
        kicker="Image satellite")

### Anatomie d'un appel d'outil

À gauche, ce que le LLM reçoit **avant** d'agir : le schéma JSON généré automatiquement à partir de la signature et de la docstring. À droite, ce qu'il reçoit **après** l'appel : un résumé compact, des notes de prudence et un `dataset_id`, mais **pas** le tableau complet.

In [ ]:
import html as html_lib
# Si la cellule précédente n'a pas été exécutée, on recalcule ce dont on a besoin.
if "stats" not in globals():
    S.new_turn()
    stats = get_admin_stats()
schema = json.dumps(TOOLS["get_admin_stats"]["schema"], ensure_ascii=False, indent=2)
result = json.dumps(stats, ensure_ascii=False, indent=2, default=str)
ui.card("Ce que voit le LLM",
        '<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">'
        f'<div><div class="ntl-kicker">① Schéma de l\'outil (avant)</div><pre style="max-height:420px">{html_lib.escape(schema)}</pre></div>'
        f'<div><div class="ntl-kicker">② Résultat renvoyé (après)</div><pre style="max-height:420px">{html_lib.escape(result[:3500])}</pre></div>'
        '</div>', kicker="Anatomie d'un appel")

## 08 · L'agent : la boucle raisonner → agir → observer

```
question ──► LLM ──► appels d'outils ? ──oui──► exécution Python ──► résultats JSON ──┐
               ▲                         │                                             │
               └─────────────────────────┼─────────────────────────────────────────────┘
                                         non
                                         ▼
                            réponse finale (Markdown) + KPI + graphiques + trace
```

Trois éléments font la qualité d'un agent :

1. **Le prompt système** : rôle, contexte (pays, données disponibles), règles (aucun chiffre sans outil, toujours mentionner les limites) et **format de réponse imposé**. C'est le « contrat de travail » de l'agent.
2. **La boucle** : jusqu'à `MAX_AGENT_STEPS` allers-retours. Les erreurs d'outils sont renvoyées au LLM pour qu'il se corrige ; un appel mal formé est redemandé.
3. **La restitution** : réponse Markdown mise en forme, indicateurs clés, graphiques et **trace complète** des appels (outil, arguments, durée) pour l'audit. Un résultat statistique doit rester vérifiable.

In [ ]:
import html as html_lib


def _clean(o):
    """Rend un résultat d'outil sérialisable et compact (arrondis, NaN → null, types numpy → Python)."""
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (float, np.floating)):
        o = float(o)
        if math.isnan(o) or math.isinf(o):
            return None
        return round(o, 3) if abs(o) < 1000 else round(o, 1)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (pd.Timestamp, dt.date)):
        return str(o)
    return o


def _coerce_args(fn, args):
    """Tolère les petites erreurs de typage des LLM ('2020' au lieu de 2020, arguments inconnus)."""
    params = inspect.signature(fn).parameters
    clean, dropped = {}, []
    for k, v in (args or {}).items():
        if k not in params:
            dropped.append(k)
            continue
        ann = params[k].annotation
        if get_origin(ann) is Union:
            ann = [a for a in get_args(ann) if a is not type(None)][0]
        if v is None or v == "":
            continue
        try:
            if ann is int:
                v = int(float(v))
            elif ann is float:
                v = float(v)
            elif ann is bool:
                v = v if isinstance(v, bool) else str(v).lower() in ("true", "1", "yes", "oui")
            elif ann is str or get_origin(ann) is Literal:
                v = str(v)
        except (TypeError, ValueError):
            pass
        clean[k] = v
    return clean, dropped


MAX_TOOL_CHARS = 7000


def execute_tool(name, args, allowed):
    t0 = time.time()
    if name not in allowed:
        out = {"ok": False, "error": f"Outil inconnu : {name}", "available_tools": sorted(allowed)}
    elif args is None:
        out = {"ok": False, "error": "Arguments JSON illisibles.", "hint": "Renvoie un objet JSON valide."}
    else:
        fn = TOOLS[name]["fn"]
        clean, dropped = _coerce_args(fn, args)
        try:
            out = {"ok": True, **fn(**clean)}
            if dropped:
                out["ignored_arguments"] = dropped
        except Exception as e:
            out = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:500]}",
                   "hint": "Corrige les arguments (années disponibles, niveau 1 ou 2, nom exact de région, "
                           "dataset_id existant) puis réessaie, ou explique la limite à l'utilisateur."}
    txt = json.dumps(_clean(out), ensure_ascii=False, default=str)
    if len(txt) > MAX_TOOL_CHARS:
        txt = txt[:MAX_TOOL_CHARS] + ' …[tronqué : utiliser le dataset_id pour la suite]"}'
    short = out.get("dataset_id") or out.get("figure_id") or out.get("path") or out.get("error", "")
    rec = dict(tool=name, args=args, ok=bool(out.get("ok")), duration=time.time() - t0, result=str(short)[:160])
    return txt, rec


def build_system_prompt(iso3=None):
    iso = _iso(iso3)
    synth = ("\n⚠️ Les données sont SYNTHÉTIQUES (mode démo) : commence ta réponse par un avertissement explicite "
             "et ne présente jamais ces chiffres comme réels.") if backend.synthetic else ""
    return f"""Tu es « Lumina », analyste géospatial senior spécialiste des lumières nocturnes (VIIRS), au service de
l'institut national de statistique de {country_name(iso)} ({iso}), dans le cadre de l'atelier STG17 (BAD / UA STATAFRIC).

CONTEXTE
- Pays par défaut : {country_name(iso)} (ISO3 {iso}). Tu peux analyser d'autres pays si on te le demande (paramètre iso3).
- Données : {backend.name}. Annuel : {min(backend.annual_years)}–{max(backend.annual_years)} ; mensuel :
  {backend.availability['monthly_months'][0]} → {backend.availability['monthly_months'][-1]}.
- Pixel « éclairé » : radiance > {LIT_THRESHOLD} nW/cm²/sr. SOL = somme des radiances d'une zone.
- Date du jour : {dt.date.today().isoformat()}.{synth}

RÈGLES
1. Aucun chiffre sans outil : chaque valeur citée provient d'un résultat d'outil de cette conversation.
2. Planifie : d'abord les outils de données, puis les graphiques (plot_*, show_night_image) avec le dataset_id
   obtenu. Produis au moins un graphique dès que la question porte sur une évolution, une comparaison ou une
   répartition spatiale (carte pour le spatial). Deux ou trois graphiques au maximum.
3. Si un outil renvoie une erreur, lis-la, corrige les arguments et réessaie (2 fois maximum), sinon explique.
4. Avant d'interpréter une baisse mensuelle, regarde la couverture nuageuse (coverage_pct ou get_quality_report).
5. Les NTL sont un PROXY de l'activité et de l'électrification : n'en déduis jamais directement le PIB, la
   pauvreté ou l'accès à l'électricité sans le dire. Signale les petits échantillons et les ruptures de source.
6. Sois concis et précis. Arrondis intelligemment (2 à 3 chiffres significatifs). Pas de tableau de plus de 8 lignes.

FORMAT DE LA RÉPONSE FINALE — Markdown, rédigé en {ANSWER_LANGUAGE} (titres traduits dans cette langue) :
### 🔎 En bref
2 à 3 phrases qui répondent directement, avec les chiffres clés en **gras**.
### 📊 Résultats
Un tableau Markdown compact ou des puces.
### 🧭 Interprétation
Ce que cela signifie pour un statisticien public (3 à 5 puces).
### ⚠️ Limites et précautions
2 à 4 puces SPÉCIFIQUES à cette analyse.
### 🛠️ Méthode
Outils, produits et datasets utilisés (1 à 2 lignes).
N'insère ni image ni lien : les graphiques s'affichent automatiquement sous ta réponse ; cite-les par leur titre."""

### L'interface de l'agent : une réponse lisible comme une note d'analyse

Une réponse d'agent destinée à un institut de statistique doit se lire comme une **note d'analyse**, pas comme un fil de discussion. Chaque réponse est donc mise en page ainsi :

| Bloc | Rôle |
|---|---|
| **En-tête** | Question posée, pays, statut des données (réelles ou démo), modèle, durée, nombre d'outils |
| **Indicateurs clés** | Les 3 à 6 chiffres essentiels, calculés par les outils (jamais par le LLM) |
| **Démarche de l'agent** | La suite des outils appelés, dans l'ordre, avec leur statut : on voit *comment* le résultat a été obtenu |
| **Note structurée** | En bref, Résultats, Interprétation, Limites et Méthode, chaque bloc ayant son propre style |
| **Figures numérotées** | Graphiques et cartes interactifs, légendés « Figure n » |
| **Pied de page** | Sources et mention « à valider avant diffusion », plus une trace d'audit dépliable |

In [ ]:
LX_CSS = """<style>
.lx{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;color:#231F20;max-width:1180px;margin:6px 0 4px;
  border-radius:16px;overflow:hidden;border:1px solid #D5DED9;background:#fff;box-shadow:0 4px 18px rgba(0,85,58,.10)}
.lx-head{position:relative;overflow:hidden;padding:20px 26px 24px;color:#fff;
  background:linear-gradient(135deg,#00553A 0%,#00704A 50%,#00A86A 100%)}
.lx-head:after{content:"";position:absolute;left:0;right:0;bottom:0;height:5px;background:#F5C242}
.lx-rings{position:absolute;right:-60px;top:-70px;opacity:.14;pointer-events:none}
.lx-top{display:flex;align-items:center;gap:12px;position:relative}
.lx-avatar{width:44px;height:44px;border-radius:50%;background:#0b2e4f;display:flex;align-items:center;justify-content:center;
  box-shadow:0 0 0 3px rgba(245,194,66,.55),0 0 18px rgba(245,194,66,.55);font-size:22px;flex:none}
.lx-kick{font-size:11px;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#F5C242}
.lx-sub{font-size:12.5px;color:#E6F6EE}
.lx-badge{margin-left:auto;font-size:12px;font-weight:700;padding:5px 12px;border-radius:999px;white-space:nowrap}
.lx-badge.real{background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.45);color:#fff}
.lx-badge.demo{background:#B83B2E;border:1px solid #fff;color:#fff}
.lx-q{position:relative;font-size:21px;font-weight:700;line-height:1.3;margin:14px 0 12px;max-width:980px}
.lx-chips{display:flex;flex-wrap:wrap;gap:6px;position:relative}
.lx-chip{font-size:12px;padding:3px 10px;border-radius:999px;background:rgba(255,255,255,.13);border:1px solid rgba(255,255,255,.3)}
.lx-body{padding:18px 26px 8px}
.lx-label{font-size:10.5px;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#00704A;margin:6px 0 8px}
.lx-kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px;margin-bottom:16px}
.lx-kpi{position:relative;background:#F4F7F5;border:1px solid #D5DED9;border-radius:12px;padding:12px 14px 10px;overflow:hidden}
.lx-kpi:before{content:"";position:absolute;left:0;top:0;bottom:0;width:4px;background:#00A86A}
.lx-kpi:nth-child(2):before{background:#00704A}.lx-kpi:nth-child(3):before{background:#0E7C86}
.lx-kpi:nth-child(4):before{background:#D49A00}.lx-kpi:nth-child(5):before{background:#C4621D}
.lx-kpi .v{font-size:25px;font-weight:700;color:#00553A;line-height:1.1;word-break:break-word}
.lx-kpi .u{font-size:12.5px;color:#5E6964;font-weight:400}
.lx-kpi .l{font-size:12px;color:#5E6964;margin-top:3px}
.lx-kpi .d{font-size:12px;font-weight:700;margin-top:2px}
.lx-up{color:#00A86A}.lx-down{color:#B83B2E}
.lx-flow{display:flex;flex-wrap:wrap;align-items:center;gap:4px;margin-bottom:18px}
.lx-step{display:inline-flex;align-items:center;gap:6px;background:#fff;border:1px solid #D5DED9;border-radius:10px;
  padding:5px 10px;font-size:12.5px}
.lx-step .n{width:18px;height:18px;border-radius:50%;background:#00704A;color:#fff;font-size:10.5px;font-weight:700;
  display:inline-flex;align-items:center;justify-content:center}
.lx-step.err{border-color:#B83B2E;background:rgba(184,59,46,.06)}.lx-step.err .n{background:#B83B2E}
.lx-step .t{color:#8A9590;font-size:11px}
.lx-arrow{color:#A9B5B0;font-size:14px}
.lx-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:14px}
.lx-sec{border:1px solid #D5DED9;border-radius:12px;padding:12px 16px;background:#fff;line-height:1.55;font-size:14.5px}
.lx-sec .h{font-size:13px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;color:#00704A;margin-bottom:6px}
.lx-sec.summary{grid-column:1/-1;background:#E8F5EF;border:1.5px solid #00A86A;font-size:16px}
.lx-sec.summary .h{color:#00553A}
.lx-sec.results{grid-column:1/-1}
.lx-sec.limits{background:rgba(245,194,66,.10);border-color:#D49A00}.lx-sec.limits .h{color:#8a6300}
.lx-sec.method{background:#F4F7F5;color:#5E6964;font-size:13px}.lx-sec.method .h{color:#5E6964}
.lx-sec p{margin:4px 0}.lx-sec ul,.lx-sec ol{margin:4px 0 4px 18px;padding:0}.lx-sec li{margin:2px 0}
.lx-sec table{border-collapse:collapse;width:100%;font-size:13.5px;margin:6px 0}
.lx-sec th{background:#E8F5EF;color:#00553A;text-align:left !important;padding:6px 10px;border-bottom:2px solid #00A86A}
.lx-sec td{padding:5px 10px;border-bottom:1px solid #E1E7E4;text-align:left !important}
.lx-sec tr:nth-child(even) td{background:#FAFCFB}
.lx-sec strong{color:#00553A}
.lx-sec code{background:#F4F7F5;padding:1px 5px;border-radius:4px;font-family:'Courier New',monospace;font-size:12.5px}
.lx-alert{background:rgba(184,59,46,.08);border:1px solid #B83B2E;color:#7a2219;border-radius:10px;padding:9px 14px;margin-bottom:14px}
.lx-foot{display:flex;flex-wrap:wrap;gap:8px 18px;justify-content:space-between;border-top:1px solid #E1E7E4;
  margin-top:16px;padding:10px 0 12px;font-size:11.5px;color:#8A9590}
.lx-figcap{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;display:flex;align-items:center;gap:10px;max-width:1180px;
  margin:14px 0 0;padding:8px 14px;border-radius:10px 10px 0 0;background:#F4F7F5;border:1px solid #D5DED9;border-bottom:3px solid #00A86A}
.lx-figcap .n{font-size:11px;font-weight:700;letter-spacing:2px;text-transform:uppercase;color:#fff;background:#00704A;
  padding:2px 9px;border-radius:999px}
.lx-figcap .t{font-weight:700;color:#231F20;font-size:14px}
.lx-trace{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;max-width:1180px;margin:10px 0 18px;font-size:13px}
.lx-trace summary{cursor:pointer;color:#00704A;font-weight:700}
.lx-trace table{border-collapse:collapse;margin-top:8px;width:100%}
.lx-trace th{background:#E8F5EF;color:#00553A;text-align:left;padding:5px 8px;border-bottom:2px solid #00A86A}
.lx-trace td{padding:4px 8px;border-bottom:1px solid #E1E7E4;text-align:left;vertical-align:top}
.lx-trace code{font-family:'Courier New',monospace;font-size:12px}
.lx-work{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;max-width:1180px;border:1px solid #D5DED9;border-radius:14px;
  overflow:hidden;background:#fff}
.lx-work .bar{height:4px;background:linear-gradient(90deg,#F5C242,#00A86A,#F5C242);background-size:200% 100%;
  animation:lxmove 1.4s linear infinite}
@keyframes lxmove{0%{background-position:0 0}100%{background-position:-200% 0}}
.lx-work .in{padding:12px 18px;display:flex;gap:14px;align-items:flex-start}
.lx-pulse{width:38px;height:38px;border-radius:50%;background:#0b2e4f;display:flex;align-items:center;justify-content:center;
  font-size:19px;flex:none;animation:lxglow 1.6s ease-in-out infinite}
@keyframes lxglow{0%,100%{box-shadow:0 0 0 2px rgba(245,194,66,.4),0 0 6px rgba(245,194,66,.4)}
  50%{box-shadow:0 0 0 4px rgba(245,194,66,.7),0 0 20px rgba(245,194,66,.8)}}
.lx-work .state{font-weight:700;font-size:14.5px;margin-bottom:6px}
</style>"""

LX_RINGS = ('<svg class="lx-rings" width="330" height="330" viewBox="0 0 330 330"><g fill="none" stroke="#fff" '
            'stroke-width="2"><circle cx="165" cy="165" r="160"/><circle cx="165" cy="165" r="118"/>'
            '<circle cx="165" cy="165" r="76"/><circle cx="165" cy="165" r="34"/></g></svg>')

FAMILY_ICON = {"Découverte": "🧭", "Séries temporelles": "📈", "Analyse spatiale": "🗺️",
               "Population & validation": "👥", "Événements & qualité": "⚡", "Visualisation": "🎨",
               "Export": "💾", "Extension": "🧩"}
TOOL_LABEL = {
    "get_country_profile": "Profil du pays", "get_data_availability": "Disponibilité des données",
    "list_admin_units": "Unités administratives", "get_national_timeseries": "Série nationale",
    "get_admin_timeseries": "Séries régionales", "detect_anomalies": "Détection d'anomalies",
    "get_admin_stats": "Statistiques régionales", "compare_periods": "Comparaison de périodes",
    "get_population_light_stats": "Population & lumière", "get_worldbank_indicator": "Indicateur Banque mondiale",
    "correlate_ntl_with_indicator": "Validation du proxy", "compare_countries": "Comparaison de pays",
    "assess_event_impact": "Impact d'événement", "get_quality_report": "Contrôle qualité",
    "plot_timeseries": "Graphique d'évolution", "plot_ranking": "Classement", "plot_map": "Carte",
    "plot_scatter": "Nuage de points", "show_night_image": "Image nocturne", "export_dataset": "Export",
    "list_session_datasets": "Datasets de la session", "get_region_profile": "Fiche région",
}
SECTION_KEYS = [("summary", ("bref", "brief", "summary", "resumo", "síntese", "🔎")),
                ("results", ("résultat", "result", "resultado", "📊")),
                ("interp", ("interprét", "interpret", "🧭")),
                ("limits", ("limite", "limit", "précaution", "caveat", "⚠")),
                ("method", ("méthode", "method", "método", "🛠"))]


def _tool_chip(name):
    fam = TOOLS.get(name, {}).get("family", "")
    return f'{FAMILY_ICON.get(fam, "🔧")} {TOOL_LABEL.get(name, name)}'


def _unique_kpis(kpis, n=6):
    seen, out = set(), []
    for k in kpis:
        if k["label"] not in seen:
            seen.add(k["label"])
            out.append(k)
    return out[:n]


def _kpi_html(kpis):
    cells = ""
    for k in kpis:
        d = k.get("delta")
        dh = ""
        if d is not None and not (isinstance(d, float) and math.isnan(d)):
            dh = f'<div class="d {"lx-up" if d >= 0 else "lx-down"}">{"▲" if d >= 0 else "▼"} {abs(d):.1f} %</div>'
        cells += (f'<div class="lx-kpi"><div class="v">{html_lib.escape(str(k["value"]))} '
                  f'<span class="u">{html_lib.escape(str(k.get("unit", "")))}</span></div>'
                  f'<div class="l">{html_lib.escape(k["label"])}</div>{dh}</div>')
    return f'<div class="lx-label">Indicateurs clés</div><div class="lx-kpis">{cells}</div>' if cells else ""


def _flow_html(trace):
    if not trace:
        return ""
    steps = []
    for i, r in enumerate(trace, 1):
        err = "" if r["ok"] else " err"
        mark = "" if r["ok"] else " ⚠"
        steps.append(f'<span class="lx-step{err}" title="{html_lib.escape(r["tool"])}"><span class="n">{i}</span>'
                     f'{_tool_chip(r["tool"])}{mark}<span class="t">{r["duration"]:.1f} s</span></span>')
    return ('<div class="lx-label">Démarche de l\'agent</div><div class="lx-flow">'
            + '<span class="lx-arrow">›</span>'.join(steps) + '</div>')


def _sections_html(answer_md):
    """Découpe la réponse Markdown en blocs (En bref, Résultats, …) et donne à chacun son style."""
    html = ui.md(answer_md)
    parts = re.split(r"<h[23][^>]*>(.*?)</h[23]>", html, flags=re.S)
    if len(parts) == 1:
        return f'<div class="lx-grid"><div class="lx-sec summary">{html}</div></div>'
    out = f'<div class="lx-sec summary">{parts[0]}</div>' if parts[0].strip() else ""
    for title, body in zip(parts[1::2], parts[2::2]):
        low = title.lower()
        cls = next((c for c, keys in SECTION_KEYS if any(k in low for k in keys)), "interp")
        clean = re.sub(r"^[^\wÀ-ÿ]+", "", re.sub(r"<[^>]+>", "", title)).strip()
        out += f'<div class="lx-sec {cls}"><div class="h">{clean}</div>{body}</div>'
    return f'<div class="lx-grid">{out}</div>'


def _header_html(kicker, sub, question, chips, synthetic):
    badge = ('<span class="lx-badge demo">● DÉMO — données synthétiques</span>' if synthetic else
             f'<span class="lx-badge real">● Données réelles · {html_lib.escape(backend.name)}</span>')
    chips_html = "".join(f'<span class="lx-chip">{c}</span>' for c in chips)
    q = f'<div class="lx-q">« {html_lib.escape(question)} »</div>' if question else '<div style="height:10px"></div>'
    return (f'<div class="lx-head">{LX_RINGS}<div class="lx-top"><div class="lx-avatar">🌙</div>'
            f'<div><div class="lx-kick">{kicker}</div><div class="lx-sub">{sub}</div></div>{badge}</div>'
            f'{q}<div class="lx-chips">{chips_html}</div></div>')


def lx_progress_html(state, trace, model, step):
    done = "".join(f'<span class="lx-step{"" if r["ok"] else " err"}"><span class="n">{i}</span>{_tool_chip(r["tool"])}'
                   f'<span class="t">{r["duration"]:.1f} s</span></span>' for i, r in enumerate(trace, 1))
    return (LX_CSS + f'<div class="lx-work"><div class="bar"></div><div class="in"><div class="lx-pulse">🌙</div><div>'
            f'<div class="state">{state}</div><div class="lx-muted" style="font-size:12px;color:#5E6964;margin-bottom:6px">'
            f'Lumina · étape {step} · modèle <code>{html_lib.escape(model)}</code></div>'
            f'<div class="lx-flow" style="margin:0">{done}</div></div></div></div>')


FIG_IN_IFRAME = False
# Dans un widget de sortie (chat, section 10), Colab n'exécute pas les scripts de Plotly :
# la figure est alors placée dans une iframe autonome, qui s'affiche partout.


def show_figure(fig):
    if not FIG_IN_IFRAME:
        return fig.show()
    h = int(fig.layout.height or 480) + 24
    doc = fig.to_html(include_plotlyjs="cdn", full_html=True, default_width="100%", config={"responsive": True})
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        display(HTML(f'<iframe srcdoc="{html_lib.escape(doc, quote=True)}" '
                     f'style="width:100%;max-width:1180px;height:{h}px;border:0;background:#fff"></iframe>'))


def render_turn(t):
    """Affiche un tour de l'agent comme une note d'analyse."""
    country = country_name(COUNTRY_ISO3)
    chips = [f"🧠 {html_lib.escape(t['provider'])} · {html_lib.escape(t['model'])}", f"⏱ {t['seconds']:.1f} s",
             f"🛠 {len(t['trace'])} outil(s)", f"🔁 {t['steps']} étape(s)"]
    if t["tokens"]:
        chips.append(f"🔤 {fmt(t['tokens'])} jetons")
    if t["figures"]:
        chips.append(f"📊 {len(t['figures'])} figure(s)")
    head = _header_html(f"Lumina · Analyste NTL · {html_lib.escape(country)} ({COUNTRY_ISO3})",
                        f"Question {t['id']} · {t['when']} · atelier STG17", t["question"], chips, t["synthetic"])
    body = ""
    if t["synthetic"]:
        body += ('<div class="lx-alert"><b>Mode démo :</b> ces chiffres sont synthétiques et servent uniquement à '
                 'apprendre la mécanique de l\'agent. Ne pas les citer.</div>')
    body += _kpi_html(_unique_kpis(t["kpis"]))
    body += _flow_html(t["trace"])
    body += '<div class="lx-label">Note d\'analyse</div>' + _sections_html(t["answer"])
    datasets = [r["result"] for r in t["trace"] if r["ok"] and r["result"].startswith("ds")]
    foot_left = ("Sources : NOAA/EOG VIIRS, NASA Black Marble, WorldPop, geoBoundaries, Banque mondiale — via "
                 "Google Earth Engine" if not t["synthetic"] else "Source : données synthétiques (mode démo)")
    foot_right = "🤖 Analyse générée par IA à partir d'outils tracés — à valider par un statisticien avant diffusion"
    if datasets:
        foot_left += f" · datasets : {', '.join(datasets)}"
    body += f'<div class="lx-foot"><span>{foot_left}</span><span>{foot_right}</span></div>'
    display(HTML(LX_CSS + f'<div class="lx">{head}<div class="lx-body">{body}</div></div>'))

    for n, fid in enumerate(t["figures"], 1):
        f = S.figures[fid]
        display(HTML(LX_CSS + f'<div class="lx-figcap"><span class="n">Figure {n}</span>'
                     f'<span class="t">{html_lib.escape(f["title"])}</span></div>'))
        if f["kind"] == "plotly":
            show_figure(f["obj"])
        else:
            display(HTML(f'<div style="max-width:1180px;background:#000814;border-radius:0 0 12px 12px;padding:14px;'
                         f'text-align:center"><img src="data:image/png;base64,{base64.b64encode(f["obj"]).decode()}" '
                         f'style="max-width:100%;max-height:560px"></div>'))

    if t["trace"]:
        rows = "".join(
            f'<tr><td>{i}</td><td>{_tool_chip(r["tool"])}<br><code>{r["tool"]}</code></td>'
            f'<td><code>{html_lib.escape(json.dumps(r["args"], ensure_ascii=False))[:200]}</code></td>'
            f'<td>{r["duration"]:.1f} s</td><td>{"✅" if r["ok"] else "⚠️"}</td>'
            f'<td style="color:#5E6964">{html_lib.escape(r["result"])}</td></tr>' for i, r in enumerate(t["trace"], 1))
        display(HTML(LX_CSS + f'<details class="lx-trace"><summary>🔍 Trace d\'audit : {len(t["trace"])} appel(s) '
                     f'd\'outils, arguments exacts et résultats</summary><table><tr><th>#</th><th>Outil</th>'
                     f'<th>Arguments</th><th>Durée</th><th>OK</th><th>Résultat</th></tr>{rows}</table></details>'))


def render_welcome(a):
    """Carte d'accueil de l'agent : identité, périmètre et capacités."""
    fams = {}
    for n in TOOLS:
        if n in a.allowed:
            fams.setdefault(TOOLS[n]["family"], []).append(n)
    chips = [f"🧠 {html_lib.escape(a.llm.provider)} · {html_lib.escape(a.llm.model)}",
             f"🧰 {len(a.allowed)} outils ({a.toolset})", f"🗣 réponses en {ANSWER_LANGUAGE}",
             f"📅 annuel {min(backend.annual_years)}–{max(backend.annual_years)}"]
    head = _header_html(f"Lumina · Analyste NTL · {html_lib.escape(country_name(a.iso3))} ({a.iso3})",
                        "Agent prêt · atelier STG17 · Africa after dark",
                        "Posez-moi une question sur les lumières nocturnes du pays : tendances, régions, anomalies, "
                        "population, validation…", chips, backend.synthetic)
    cards = "".join(
        f'<div class="lx-sec"><div class="h">{FAMILY_ICON.get(f, "🔧")} {f}</div>'
        f'{", ".join(TOOL_LABEL.get(n, n) for n in ns)}</div>' for f, ns in fams.items())
    display(HTML(LX_CSS + f'<div class="lx">{head}<div class="lx-body"><div class="lx-label">Ce que je sais faire</div>'
                 f'<div class="lx-grid" style="grid-template-columns:repeat(auto-fit,minmax(230px,1fr))">{cards}</div>'
                 f'<div class="lx-foot"><span>Chaque chiffre provient d\'un outil tracé ; chaque réponse inclut ses limites.'
                 f'</span><span>Essayez : <code>ask("Quelles régions ont le plus progressé depuis 2015 ?")</code></span>'
                 f'</div></div></div>'))

In [ ]:
class NTLAgent:
    def __init__(self, llm, toolset=TOOLSET, iso3=None, max_steps=MAX_AGENT_STEPS, memory_turns=3):
        self.llm, self.iso3, self.max_steps, self.memory_turns = llm, _iso(iso3), max_steps, memory_turns
        self.set_toolset(toolset)
        self.memory = []

    def set_toolset(self, toolset):
        self.toolset = toolset
        self.allowed = {n for n, t in TOOLS.items() if toolset == "full" or t["core"]}
        self.schemas = [TOOLS[n]["schema"] for n in TOOLS if n in self.allowed]

    def reset(self):
        self.memory = []
        UI.ok("Nouvelle conversation : la mémoire de l'agent est vidée.")

    def _assistant_message(self, msg, step):
        calls = []
        for i, c in enumerate(msg.tool_calls):
            tc = {"id": c.id or f"call_{step}_{i}", "type": "function",
                  "function": {"name": c.function.name, "arguments": c.function.arguments or "{}"}}
            extra = (getattr(c, "model_extra", None) or {}).get("extra_content")
            if extra and self.llm.provider == "gemini":        # signature de pensée Gemini à renvoyer telle quelle
                tc["extra_content"] = extra
            calls.append(tc)
        return {"role": "assistant", "content": msg.content or None, "tool_calls": calls}

    def ask(self, question, show=True):
        S.new_turn()
        fig_before = set(S.figures)
        messages = [{"role": "system", "content": build_system_prompt(self.iso3)}]
        for q, a in self.memory[-self.memory_turns:]:
            messages += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
        messages.append({"role": "user", "content": question})
        trace, tokens, t0, answer, fixes = [], 0, time.time(), "", 0
        live = display(HTML(""), display_id=True) if show else None

        def progress(state):
            if live is not None:
                live.update(HTML(lx_progress_html(state, trace, self.llm.model, step)))

        step = 0
        while step < self.max_steps:
            step += 1
            progress("Je réfléchis à la meilleure démarche…" if step == 1 else "J'analyse les résultats obtenus…")
            try:
                resp = self.llm.chat(messages, tools=self.schemas)
            except openai.BadRequestError as e:
                if fixes < 2 and ("tool" in str(e).lower() or "function" in str(e).lower()):
                    fixes += 1
                    messages.append({"role": "user", "content": "Ton dernier appel d'outil était mal formé. "
                                     "Réessaie en respectant exactement le schéma JSON des paramètres."})
                    continue
                raise
            if getattr(resp, "usage", None):
                tokens += (resp.usage.prompt_tokens or 0) + (resp.usage.completion_tokens or 0)
            msg = resp.choices[0].message
            if not msg.tool_calls:
                answer = (msg.content or "").strip()
                if answer:
                    break
                messages.append({"role": "user", "content": "Rédige maintenant la réponse finale au format demandé."})
                continue
            messages.append(self._assistant_message(msg, step))
            for tc in messages[-1]["tool_calls"]:
                name, raw = tc["function"]["name"], tc["function"]["arguments"]
                try:
                    args = json.loads(raw) if raw and raw.strip() else {}
                except json.JSONDecodeError:
                    args = None
                progress(f"J'exécute : {_tool_chip(name)}…")
                result, rec = execute_tool(name, args, self.allowed)
                rec["step"] = step
                trace.append(rec)
                messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
        if not answer:
            messages.append({"role": "user", "content": "Nombre maximal d'étapes atteint : rédige la réponse finale "
                             "avec les résultats déjà obtenus, au format demandé."})
            answer = (self.llm.chat(messages).choices[0].message.content or "").strip() or "_(pas de réponse)_"

        turn = dict(id=S.turn_id, question=question, answer=answer, trace=trace, kpis=list(S.kpis),
                    figures=[f for f in S.figures if f not in fig_before], provider=self.llm.provider,
                    model=self.llm.model, seconds=time.time() - t0, steps=step, tokens=tokens,
                    toolset=self.toolset, synthetic=backend.synthetic, when=dt.datetime.now().strftime("%Y-%m-%d %H:%M"))
        S.turns.append(turn)
        self.memory.append((question, answer))
        if live is not None:
            live.update(HTML(""))
        if show:
            render_turn(turn)
        return turn


agent = NTLAgent(llm) if llm else None


def ask(question, **kw):
    """Raccourci : pose une question à l'agent de la session."""
    if agent is None:
        return ui.warn("Aucun LLM connecté : revenez à la section 04.")
    agent.ask(question, **kw)          # le tour complet reste accessible dans S.turns[-1]


if agent:
    render_welcome(agent)
else:
    ui.warn("Agent non créé : aucun LLM connecté (section 04).")

In [ ]:
# 👀 Curieux ? Voici le prompt système exact envoyé au LLM.
ui.card("Prompt système", f"<pre>{html_lib.escape(build_system_prompt())}</pre>", kicker="Sous le capot")

## 09 · Démonstrations guidées

Chaque cellule pose une question de difficulté croissante. Observez dans la **trace** quels outils l'agent choisit, dans quel ordre, et s'il se corrige après une erreur. Modifiez librement les questions.

> ⏱️ Avec Earth Engine, le premier calcul d'un pays prend de quelques secondes à quelques minutes selon sa taille ; les résultats sont ensuite **mis en cache** (`ntl_agent_outputs/cache`).

In [ ]:
# 1️⃣ Tendance nationale
ask("Comment la luminosité nocturne nationale a-t-elle évolué sur les dix dernières années disponibles ? "
    "Donne la tendance, le taux de croissance annuel moyen et un graphique.")

In [ ]:
# 2️⃣ Géographie de la lumière
ask("Quelles sont les régions (ADM1) les plus et les moins éclairées lors de la dernière année disponible ? "
    "Montre une carte de la radiance moyenne et un classement, et dis si la lumière est très concentrée.")

In [ ]:
# 3️⃣ Dynamique infranationale
ask("Quelles régions ont le plus progressé entre 2015 et la dernière année disponible ? "
    "Les régions les moins éclairées rattrapent-elles les autres ? Carte des variations à l'appui.")

In [ ]:
# 4️⃣ Anomalies mensuelles — vraies baisses ou nuages ?
ask("Y a-t-il eu des mois anormaux ces quatre dernières années ? Distingue les baisses probablement réelles "
    "des artefacts liés aux nuages, avec un graphique.")

In [ ]:
# 5️⃣ Validation du proxy (Jour 4, après-midi)
ask("Les lumières nocturnes sont-elles un bon proxy du PIB dans ce pays ? Valide avec les données de la Banque "
    "mondiale et conclus : utilisable pour la diffusion ou seulement comme outil de diagnostic ?")

In [ ]:
# 6️⃣ Population dans l'obscurité et comparaison avec des pays pairs
PEERS = "KEN,UGA,TZA,BDI"   # ← adaptez aux voisins de votre pays
ask(f"Quelle part de la population vit dans des zones non éclairées, par région ? Puis compare {COUNTRY_ISO3} "
    f"avec {PEERS} en SOL par habitant et en accès à l'électricité.")

In [ ]:
# 7️⃣ Impact d'un événement daté (Black Marble journalier) — renseignez une date qui a du sens pour votre pays
EVENT_DATE = "2023-05-03"   # ← ex. inondation, coupure nationale, cyclone, conflit
ask(f"Évalue l'impact sur l'éclairage nocturne de l'événement du {EVENT_DATE} (fenêtres de 30 jours avant/après), "
    "par région, avec une carte.")

## 10 · Discutez avec l'agent

Une interface de conversation : saisissez votre question ou cliquez sur une suggestion. L'agent garde en mémoire les trois derniers échanges, vous pouvez donc enchaîner (« et pour les districts ? », « fais-en une carte »).
Si les widgets ne s'affichent pas dans votre environnement, utilisez simplement `ask("votre question")`.

In [ ]:
import ipywidgets as W

SUGGESTIONS = [
    "Montre-moi le pays de nuit pour la dernière année disponible",
    "Classe les districts (ADM2) par surface éclairée",
    "Fais un rapport qualité des données pour l'an dernier",
    "Compare 2016 et la dernière année au niveau des districts",
    "Quelle est la corrélation avec l'accès à l'électricité ?",
    "Exporte le dernier tableau en Excel",
]


def chat_ui():
    if agent is None:
        return ui.warn("Aucun LLM connecté (section 04).")
    box = W.Textarea(placeholder="Ex. : Quelles régions ont le plus progressé depuis 2015 ? Montre une carte.",
                     layout=W.Layout(width="100%", height="80px", margin="10px 0 6px"))
    send = W.Button(description="Demander à Lumina", icon="paper-plane", layout=W.Layout(width="210px", height="38px"))
    if "text_color" in send.style.keys:          # ipywidgets ≥ 8 : couleurs de la charte
        send.style.button_color, send.style.text_color, send.style.font_weight = "#00704A", "white", "bold"
    else:                                          # ipywidgets 7 (anciennes versions de Colab)
        send.button_style = "success"
    new = W.Button(description="Nouvelle conversation", icon="refresh", layout=W.Layout(width="210px", height="38px"))
    out = W.Output()
    chips = []
    for s in SUGGESTIONS:
        b = W.Button(description="✦ " + s[:46] + ("…" if len(s) > 46 else ""), tooltip=s,
                     layout=W.Layout(width="auto", margin="3px"))
        b.style.button_color = "#E8F5EF"
        b.on_click(lambda _b, s=s: setattr(box, "value", s))
        chips.append(b)

    def on_send(_):
        q = box.value.strip()
        if not q:
            return
        global FIG_IN_IFRAME
        send.disabled = True
        FIG_IN_IFRAME = True
        with out:
            try:
                agent.ask(q)
            except Exception as e:
                ui.alert(f"Erreur : <code>{html_lib.escape(str(e)[:400])}</code>")
        FIG_IN_IFRAME = False
        send.disabled = False
        box.value = ""

    def on_new(_):
        with out:
            out.clear_output()
            agent.reset()

    send.on_click(on_send)
    new.on_click(on_new)
    header = W.HTML(LX_CSS + '<div class="lx" style="margin-bottom:0">' + _header_html(
        f"Lumina · Conversation · {html_lib.escape(country_name(COUNTRY_ISO3))} ({COUNTRY_ISO3})",
        "Tapez votre question ou choisissez une suggestion — l'agent garde en mémoire les 3 derniers échanges",
        "", [f"🧠 {html_lib.escape(agent.llm.model)}", f"🧰 {len(agent.allowed)} outils",
             f"🗣 {ANSWER_LANGUAGE}"], backend.synthetic) + '</div>')
    display(W.VBox([header, W.HBox(chips, layout=W.Layout(flex_flow="row wrap")), box, W.HBox([send, new]), out]))


chat_ui()

## 11 · Laboratoire : comparer les fournisseurs

Même question, même boîte à outils, fournisseurs différents : on mesure **latence**, **nombre d'étapes**, **jetons consommés** et **réussite**. Seuls les fournisseurs dont la clé est disponible (ou Ollama s'il tourne) sont testés — aucune clé n'est demandée ici.
Reportez vos résultats dans la feuille de comparaison partagée de l'atelier (Jour 2 · *Choosing your engine*).

In [ ]:
BENCH_QUESTION = ("Quelle est la variation de la somme des radiances nationale entre 2016 et la dernière année "
                  "disponible, et le taux de croissance annuel moyen ?")


def provider_available(p):
    if p == "ollama":
        try:
            return requests.get(PROVIDERS["ollama"]["base_url"].replace("/v1", "/api/tags"), timeout=2).ok
        except Exception:
            return False
    return bool(get_secret(PROVIDERS[p]["key_env"], prompt=False))


def run_benchmark(question=BENCH_QUESTION, providers=("groq", "gemini", "ollama"), toolset="core"):
    rows = []
    for p in providers:
        if not provider_available(p):
            rows.append(dict(fournisseur=p, modèle="—", statut="non configuré"))
            continue
        try:
            a = NTLAgent(LLMClient(p, prompt_for_key=False), toolset=toolset)
            t = a.ask(question, show=False)
            rows.append(dict(fournisseur=p, modèle=t["model"], statut="✅", latence_s=round(t["seconds"], 1),
                             étapes=t["steps"], appels_outils=len(t["trace"]), jetons=t["tokens"],
                             erreurs_outils=sum(not r["ok"] for r in t["trace"]),
                             réponse=t["answer"][:160].replace("\n", " ") + "…"))
        except Exception as e:
            rows.append(dict(fournisseur=p, modèle=PROVIDERS[p]["default_model"], statut=f"❌ {str(e)[:80]}"))
    df = pd.DataFrame(rows)
    ui.card("Comparaison des fournisseurs", df.to_html(index=False, na_rep="—", escape=True), kicker="11 · Benchmark")
    ok = df[df.statut == "✅"] if "latence_s" in df else pd.DataFrame()
    if len(ok):
        fig = go.Figure(go.Bar(x=ok.fournisseur, y=ok.latence_s, marker_color=SERIES[:len(ok)],
                               text=[f"{v:.1f} s" for v in ok.latence_s], textposition="outside"))
        _style(fig, "Temps de réponse de l'agent par fournisseur", "secondes", None, height=380).show()
    return df

bench = run_benchmark()

## 12 · Mini-banc d'évaluation

« Ça a l'air juste » ne suffit pas. On construit un petit **jeu d'évaluation** : pour chaque question, la **vérité terrain est calculée par Python** avec les mêmes outils, puis on vérifie que la réponse de l'agent (a) a appelé un outil pertinent et (b) contient la bonne valeur. C'est la démarche présentée au Jour 2 (*Prompt optimisation*) : mesurer avant d'optimiser.

In [ ]:
def _truth_top_region():
    S.new_turn()
    return get_admin_stats()["concentration"]["top1_unit"]


def _truth_n_adm1():
    return str(len(backend.admin_units(COUNTRY_ISO3, 1)))


def _truth_last_year():
    return str(backend.default_year())


EVAL_SET = [
    dict(question="Combien d'unités administratives de niveau 1 compte le pays ? Réponds brièvement.",
         tools={"get_country_profile", "list_admin_units"}, truth=_truth_n_adm1),
    dict(question="Quelle région était la plus lumineuse (SOL) lors de la dernière année disponible ? Réponds brièvement.",
         tools={"get_admin_stats"}, truth=_truth_top_region),
    dict(question="Quelle est la dernière année de données annuelles disponible ? Réponds brièvement.",
         tools={"get_country_profile", "get_data_availability", "get_national_timeseries"}, truth=_truth_last_year),
]


def run_eval(a=None):
    a = a or agent
    if a is None:
        return ui.warn("Aucun LLM connecté.")
    rows = []
    for case in EVAL_SET:
        truth = case["truth"]()
        t = a.ask(case["question"], show=False)
        a.memory.clear()                      # questions indépendantes
        used = {r["tool"] for r in t["trace"]}
        norm = lambda s: re.sub(r"\s+", " ", s.lower())
        rows.append(dict(question=case["question"][:70] + "…", vérité=truth,
                         outil_pertinent="✅" if used & case["tools"] else "❌",
                         valeur_correcte="✅" if norm(truth) in norm(t["answer"]) else "❌",
                         outils=", ".join(sorted(used)) or "aucun", latence_s=round(t["seconds"], 1)))
    df = pd.DataFrame(rows)
    score = (df.outil_pertinent.eq("✅").mean() + df.valeur_correcte.eq("✅").mean()) / 2
    ui.card(f"Score : {score:.0%}", df.to_html(index=False, escape=True), kicker="12 · Évaluation",
            highlight=score >= 0.8)
    return df

eval_df = run_eval()

## 13 · Publier : un rapport HTML autonome

Toutes les questions, réponses, graphiques interactifs et traces de la session sont assemblés dans **un seul fichier HTML** : à partager par courriel ou à publier directement sur **GitHub Pages** (Jour 5). Le rapport rappelle les sources et les limites.

In [ ]:
def generate_report(path=None, title=None):
    path = Path(path or OUT / f"rapport_ntl_{COUNTRY_ISO3}_{dt.date.today():%Y%m%d}.html")
    title = title or f"Lumières nocturnes — {country_name(COUNTRY_ISO3)}"
    first_plotly = True
    sections = ""
    for t in S.turns:
        figs = ""
        for fid in t["figures"]:
            f = S.figures[fid]
            if f["kind"] == "plotly":
                figs += f["obj"].to_html(full_html=False, include_plotlyjs="cdn" if first_plotly else False)
                first_plotly = False
            else:
                figs += (f'<div style="background:#000814;border-radius:10px;padding:12px;text-align:center;margin:10px 0">'
                         f'<img src="data:image/png;base64,{base64.b64encode(f["obj"]).decode()}" style="max-width:100%"></div>')
        trace = "".join(f'<li><code>{r["tool"]}</code> {html_lib.escape(json.dumps(r["args"], ensure_ascii=False))[:160]}'
                        f' — {r["duration"]:.1f} s {"✅" if r["ok"] else "⚠️"}</li>' for r in t["trace"])
        sections += (f'<div class="lx" style="margin:18px 0 0">'
                     + _header_html(f"Lumina · Question {t['id']}", f"{t['when']} · {t['provider']} · {t['model']}",
                                    t["question"], [f"⏱ {t['seconds']:.1f} s", f"🛠 {len(t['trace'])} outil(s)"],
                                    t["synthetic"])
                     + f'<div class="lx-body">{_kpi_html(_unique_kpis(t["kpis"]))}{_flow_html(t["trace"])}'
                     f'<div class="lx-label">Note d\'analyse</div>{_sections_html(t["answer"])}'
                     f'<div class="lx-label" style="margin-top:14px">Figures</div>{figs}'
                     f'<details class="lx-trace"><summary>Trace des outils</summary><ul>{trace}</ul></details>'
                     f'</div></div>')
    if not sections:
        sections = '<div class="ntl-warn">Aucune question posée pour l\'instant.</div>'
    warn = ('<div class="ntl-alert"><b>Données synthétiques (mode démo)</b> — rapport de démonstration, '
            'chiffres non réels.</div>') if backend.synthetic else ""
    doc = f"""<!doctype html><html lang="fr"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>{html_lib.escape(title)}</title>{BASE_CSS}{LX_CSS}
<style>body{{margin:0;background:#F4F7F5}} .wrap{{max-width:1050px;margin:0 auto;padding:0 18px 40px}}</style></head>
<body class="ntl"><div style="background:linear-gradient(135deg,#00553A,#00704A 45%,#00A86A);color:#fff;
padding:44px 20px 50px;position:relative"><div class="wrap" style="padding-bottom:0">
<div style="letter-spacing:3px;font-size:12px;font-weight:700;color:#F5C242">STG17 · RAPPORT D'ANALYSE NTL</div>
<div style="font-size:38px;font-weight:700;margin:8px 0">{html_lib.escape(title)}</div>
<div style="font-style:italic;color:#E6F6EE">Produit par un agent IA ({llm.provider if llm else '—'} ·
{llm.model if llm else '—'}) · données : {backend.name} · {dt.date.today():%d/%m/%Y}</div></div>
<div style="position:absolute;left:0;right:0;bottom:0;height:7px;background:#F5C242"></div></div>
<div class="wrap">{warn}{sections}
<div class="ntl-card"><div class="ntl-kicker">Méthode, sources et limites</div>
<p>Radiances VIIRS (NOAA/EOG VNL v2 annuel, VCMSLCFG mensuel), NASA Black Marble VNP46A2, WorldPop Global2 100 m (2015–2030),
geoBoundaries v6, Banque mondiale WDI — traitement via Google Earth Engine. Seuil « éclairé » : {LIT_THRESHOLD} nW/cm²/sr.</p>
<ul><li>Les lumières nocturnes sont un <b>proxy</b> : elles ne mesurent directement ni le PIB, ni la pauvreté, ni
l'accès à l'électricité.</li><li>Artefacts possibles : nuages, saisonnalité, torchères, halo urbain, bruit rural,
ruptures de produit.</li><li>Chaque réponse a été générée par un LLM à partir des résultats d'outils tracés ci-dessus ;
elle doit être <b>vérifiée par un statisticien</b> avant toute diffusion officielle.</li></ul></div></div></body></html>"""
    path.write_text(doc, encoding="utf-8")
    ui.ok(f"Rapport enregistré : <code>{path}</code> ({path.stat().st_size / 1024:.0f} Ko, {len(S.turns)} question(s)).")
    if IN_COLAB:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception:
            pass
    return path

report_path = generate_report()

## 14 · Étendre la boîte à outils

Ajouter une compétence à l'agent = écrire **une fonction Python bien documentée** et la décorer avec `@tool`. Exemple : une **fiche synthétique d'une région** (rang, part du SOL national, évolution sur 5 ans). Recréez ensuite l'agent pour qu'il voie le nouvel outil.

**Idées d'exercices** : extension de la surface urbaine (pixels > 10 nW), détection des zones nouvellement éclairées entre deux années, indice d'inégalité intra-régionale au niveau pixel, comparaison avec les résultats d'un recensement, série DMSP-OLS historique (1992–2013) avec avertissement de rupture de capteur.

In [ ]:
@tool("Extension")
def get_region_profile(region: str, level: int = 1, iso3: Optional[str] = None) -> dict:
    """Fiche synthétique d'UNE région : rang national, part du SOL national, radiance moyenne, part éclairée et
    évolution sur les 5 dernières années. Le nom de la région est rapproché automatiquement du nom officiel.

    Args:
        region: Nom de la région (tolère fautes et accents).
        level: Niveau administratif : 1 = régions, 2 = districts.
        iso3: Code ISO3 du pays. Par défaut : le pays de la session.
    """
    iso, level = _iso(iso3), _level(level)
    last = backend.default_year()
    cur = backend.zonal_stats(iso, level, [last - 5, last])
    names = _match_units([region], cur.unit.unique())
    if not names:
        raise ValueError(f"Région « {region} » introuvable. Exemples : {list(cur.unit.unique()[:6])}")
    now = cur[cur.year == last].copy()
    now["sol_share"] = 100 * now.sol / now.sol.sum()
    now["rank"] = now.sol.rank(ascending=False).astype(int)
    r = now[now.unit == names[0]].iloc[0]
    past = cur[(cur.year == last - 5) & (cur.unit == names[0])].sol.iloc[0]
    S.add_kpi(f"Rang de {names[0]}", f"{r['rank']}/{len(now)}")
    S.add_kpi("Part du SOL national", f"{r.sol_share:.1f}", "%")
    return {"region": names[0], "year": last, "rank": int(r["rank"]), "of": len(now), "sol": r.sol,
            "sol_share_pct": r.sol_share, "mean_rad": r.mean_rad, "lit_share_pct": r.lit_share,
            "sol_change_5y_pct": 100 * (r.sol / past - 1) if past else None, "notes": _base_notes()}


if llm:
    agent = NTLAgent(llm, toolset="full")       # l'agent voit maintenant l'outil get_region_profile
    example_region = backend.admin_units(COUNTRY_ISO3, 1).unit.iloc[0]
    ask(f"Fais-moi la fiche de la région {example_region}.")

## 15 · Limites, éthique et bonnes pratiques

| Enjeu | Ce que fait ce notebook | Ce qui reste de votre responsabilité |
|---|---|---|
| **Hallucinations** | Aucun chiffre sans outil ; trace complète ; évaluation automatique | Relire chaque réponse avant diffusion ; vérifier les chiffres clés dans les datasets exportés |
| **Validité du proxy** | Outil de corrélation, notes de prudence systématiques | Valider au niveau **infranational** avec vos statistiques officielles et documenter la décision « diffusion ou diagnostic » |
| **Confidentialité** | Seuls des agrégats publics sont envoyés au LLM | Ne jamais envoyer de microdonnées à un LLM cloud ; préférer Ollama pour les données sensibles |
| **Licences** | Sources citées dans chaque graphique et dans le rapport | Respecter les licences : VIIRS/EOG (CC BY 4.0), WorldPop (CC BY 4.0), geoBoundaries (licences par pays), Banque mondiale (CC BY 4.0) |
| **Reproductibilité** | Paramètres centralisés, cache, datasets exportables, modèle et date dans le rapport | Versionner le notebook et les sorties sur GitHub ; noter la version du modèle LLM (les modèles changent) |
| **Coûts et quotas** | Cache disque, jeu d'outils `core`, reprise sur 429 | Suivre les quotas Earth Engine et LLM ; estimer le coût par analyse avant un passage à l'échelle |

### Pour aller plus loin

* Elvidge C.D. *et al.* (2021), *Annual time series of global VIIRS nighttime lights derived from monthly averages: 2012 to 2019*, **Remote Sensing** 13(5) — produit VNL v2.
* Román M.O. *et al.* (2018), *NASA's Black Marble nighttime lights product suite*, **Remote Sensing of Environment** 210.
* Runfola D. *et al.* (2020), *geoBoundaries: A global database of political administrative boundaries*, **PLoS ONE** 15(4).
* Catalogue Earth Engine : <https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_ANNUAL_V22> · <https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG> · <https://developers.google.com/earth-engine/datasets/catalog/NASA_VIIRS_002_VNP46A2>
* Banque mondiale, *Light Every Night* et le guide *Open Nighttime Lights* : <https://worldbank.github.io/OpenNightLights/>
* Compatibilité OpenAI : Groq <https://console.groq.com/docs/openai> · Gemini <https://ai.google.dev/gemini-api/docs/openai> · Ollama <https://github.com/ollama/ollama/blob/main/docs/openai.md>

---
*Notebook pédagogique réalisé pour l'atelier technique STG17 « Emerging Issues, Emerging Practice ». Charte graphique inspirée de la BAD, non officielle.*